# EF-785.raw 页岩数据的 PM-GNN 预测 notebook

本 notebook 用已经训练好的 PM-GNN，对新的页岩三维 RAW 数据进行预测：

- 数据路径：`H:\Deepl_Project\GNN\ROCK_data\test\EF-78.raw`
- 数据尺寸：`500 × 500 × 500`，代码中按 `shape_zyx = (500, 500, 500)` 读取
- voxel size：`2.25 μm × 2.25 μm × 2.25 μm`
- 孔隙阈值：`50`，默认灰度 `<= 50` 为孔隙
- 模型目录：`E:\Doctor\论文撰写\GNN\图\数据\消融实验结果\Final-PM-GNN`

运行流程：

1. 将整块 `600³` 页岩 RAW 数据裁剪为 PM-GNN 可处理的 `200³` 子块；
2. 自动加入边界子块，尽量覆盖整块样品；
3. 生成孔隙、PNM、渗透率代理标签、力学代理标签与 PM-GNN 图输入；
4. 加载 `Final-PM-GNN` 中已经训练好的 scaler 和 PM-GNN 权重，不重新训练；
5. 输出六个目标参数预测结果、指标表、六张 `true vs. predicted` 图及对应 CSV 绘图数据。

> 注意：如果该外部页岩没有独立实验/数值模拟真值，notebook 中的 `true vs. predicted` 是基于数据制作流程生成的代理标签与 PM-GNN 预测值对比；真正外部验证仍需要真实物性标签。


> **修订说明**：本版本保留 `joblib.load()` 优先读取 scaler 的修复，并加入边界子块覆盖，适合 `500³` 页岩外部数据预测。

In [ ]:

# ============================================================
# 0. 导入依赖与通用设置
# ============================================================
from pathlib import Path
import os
import math
import json
import pickle
import random
import warnings
import contextlib
import time

# 降低并行库抢占 CPU；需要在 numpy/scipy/tensorflow 导入前设置。
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")
os.environ.setdefault("TF_GPU_ALLOCATOR", "cuda_malloc_async")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from tqdm.auto import tqdm
from joblib import Parallel, delayed
import joblib

from scipy import ndimage as ndi
from scipy import sparse
from scipy.sparse.linalg import cg, spsolve
from scipy.stats import pearsonr

from skimage.morphology import remove_small_objects
from skimage.measure import block_reduce
try:
    from skimage.morphology import skeletonize_3d as _skimage_skeletonize_3d
    def skeletonize_3d_compatible(binary_mask):
        return _skimage_skeletonize_3d(binary_mask).astype(bool)
except Exception:
    from skimage.morphology import skeletonize as _skimage_skeletonize
    def skeletonize_3d_compatible(binary_mask):
        try:
            return _skimage_skeletonize(binary_mask.astype(bool), method="lee").astype(bool)
        except TypeError:
            return _skimage_skeletonize(binary_mask.astype(bool)).astype(bool)

import porespy as ps
import openpnm as op

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error,
    explained_variance_score, median_absolute_error, max_error
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# 图件风格与原 PM-GNN notebook 保持一致。
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['font.sans-serif'] = ['Times New Roman']
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['axes.unicode_minus'] = False

# TensorFlow GPU 显存按需增长。
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    """让 joblib.Parallel 在 Jupyter 中显示 tqdm 进度条。"""
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)
    old_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_callback
        tqdm_object.close()

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


In [ ]:
# ============================================================
# 1. 路径与全局参数：EF-78 页岩 RAW 数据
# ============================================================
# GitHub portable configuration (only I/O paths changed; modelling code retained).
# Launch Jupyter from the repository root, or set PMGNN_PROJECT_ROOT first.
PROJECT_ROOT = Path(os.environ.get("PMGNN_PROJECT_ROOT", str(Path.cwd()))).expanduser().resolve()
RAW_PATH = Path(os.environ.get("PMGNN_EF78_RAW", str(PROJECT_ROOT / "data" / "external" / "EF-78.raw"))).expanduser().resolve()
MODEL_DIR = Path(os.environ.get("PMGNN_MODEL_DIR", str(PROJECT_ROOT / "checkpoints" / "PM-GNN_saved_tag_9train_2val"))).expanduser().resolve()
# 输出目录：不会覆盖原始 raw，也不会覆盖训练结果目录。
DATASET_PREFIX = "EF-78"
BASE_OUT = Path(os.environ.get("PMGNN_OUTPUT_ROOT", str(PROJECT_ROOT / "outputs" / "EF-78"))).expanduser().resolve()
RAW_OUT = BASE_OUT / "raw"
PORE_OUT = BASE_OUT / "pore"
PERM_OUT = BASE_OUT / "Permeation"
MECH_OUT = BASE_OUT / "Mechanics"
SUMMARY_OUT = BASE_OUT / "summary"
PRED_OUT = BASE_OUT / "PM_GNN_prediction_results"
FIG_DATA_OUT = PRED_OUT / "figure_data"

for p in [RAW_OUT, PORE_OUT, PERM_OUT, MECH_OUT, SUMMARY_OUT, PRED_OUT, FIG_DATA_OUT]:
    p.mkdir(parents=True, exist_ok=True)

# 原始 RAW 数据参数。600×600×600 为各向同性，因此 Z/Y/X 顺序不影响尺寸校验。
SHAPE_ZYX = (600, 600, 600)
DTYPE = "auto"       # 可改为 np.uint8 / np.uint16 / np.float32 等；默认自动按文件大小推断

# 体素尺寸，单位 μm。
VOXEL_SIZE_UM = 2.25
VX_UM = VY_UM = VZ_UM = VOXEL_SIZE_UM

# 子块参数：与 PM-GNN 训练数据制作流程保持一致。
# 600³ 数据用 200³ 子块时，普通 0,200 起点会漏掉边界 100 voxel；因此开启 INCLUDE_EDGE_BLOCKS。
BLOCK_SIZE = 200
STRIDE = 100
INCLUDE_EDGE_BLOCKS = True
MAX_BLOCKS = None    # 调试时可先设为 5 或 10；正式预测保持 None。500³ 数据完整覆盖约 27 个子块。

# 孔隙阈值：EF-78 的孔隙阈值为 50，默认 block <= 90 为孔隙。
PORE_THRESHOLD = 70
PORE_MODE = "le"     # "le": block <= threshold；"equal": block == threshold
MIN_OBJECT_SIZE = 2

# 固体相：与孔隙相互补。灰度 CT 外部数据建议直接用 ~pore_mask 作为固体相。
SOLID_MIN = None
SOLID_MAX = None

# PoreSpy / OpenPNM 参数
SNOW_R_MAX = 4
SNOW_SIGMA = 0.4
SNOW_BOUNDARY_WIDTH = 3
PIN = 1.0
POUT = 0.0
VISCOSITY = 1.0

# 力学代理参数：保持与数据制作 notebook 一致
MATRIX_E_GPA = 70.0
MATRIX_NU = 0.17
MECH_MAX_SIZE = 50
MECH_SOLID_THRESHOLD = 0.5
ENABLE_SOLID_SKELETON = True
SKELETON_MAX_SIZE = 100
SKELETON_NEIGHBORHOOD = 26

# 并行设置：PNM 和力学计算较耗内存，如遇到内存不足，改成 False 或减小任务数。
CPU_USAGE_RATIO = 0.8
N_JOBS_DATASET = max(1, int((os.cpu_count() or 1) * CPU_USAGE_RATIO))
PARALLEL_PERMEATION = True
N_JOBS_PERMEATION = max(1, min(N_JOBS_DATASET, 8))
PARALLEL_MECHANICS = True
N_JOBS_MECHANICS = max(1, min(N_JOBS_DATASET, 12))
SAVE_INDIVIDUAL_MECH_XLSX = True
SAVE_INDIVIDUAL_PERMEABILITY_XLSX = True

# 如果之前已经生成过数据文件，建议保持 True，避免重复计算。
SKIP_EXISTING_FILES = True
FORCE_REBUILD_DATASET = False

BETWEENNESS_APPROX_K = 200
CENTRALITY_MAX_NODES_EXACT = 300

DIRECTION_AXIS_MASK = {"x": 2, "y": 1, "z": 0}   # mask 顺序 Z, Y, X
DIRECTION_AXIS_PN = {"x": 0, "y": 1, "z": 2}     # OpenPNM coords 顺序 X, Y, Z

print("RAW_PATH:", RAW_PATH)
print("MODEL_DIR:", MODEL_DIR)
print("BASE_OUT:", BASE_OUT)
print("SHAPE_ZYX:", SHAPE_ZYX)
print("VOXEL_SIZE_UM:", VOXEL_SIZE_UM)
print("PORE_THRESHOLD:", PORE_THRESHOLD, "; PORE_MODE:", PORE_MODE)
print("并行任务数: permeation =", N_JOBS_PERMEATION, "; mechanics =", N_JOBS_MECHANICS)


In [ ]:

# ============================================================
# 2. RAW 裁剪与孔隙分割函数
# ============================================================

def safe_num_voxels(shape_zyx):
    """安全计算体素总数，避免 NumPy 在大 RAW 尺寸下发生整数溢出。"""
    return math.prod(int(v) for v in tuple(shape_zyx))


def infer_raw_dtype(raw_path: Path, shape_zyx):
    nvox = safe_num_voxels(shape_zyx)
    size = int(raw_path.stat().st_size)
    candidates = [np.uint8, np.int8, np.uint16, np.int16, np.float32, np.uint32, np.int32, np.float64]
    matched = [dt for dt in candidates if nvox * np.dtype(dt).itemsize == size]
    if not matched:
        raise ValueError(
            f"无法根据文件大小自动推断 dtype。\n"
            f"RAW 实际大小: {size} bytes; 体素数: {nvox}; 请手动设置 DTYPE。"
        )
    # 灰度 CT / 标签数据通常优先 uint8。
    return np.dtype(matched[0])


def resolve_dtype(dtype_setting):
    if str(dtype_setting).lower() == "auto":
        return infer_raw_dtype(RAW_PATH, SHAPE_ZYX)
    return np.dtype(dtype_setting)


def check_raw_file(raw_path: Path, shape_zyx, dtype):
    if not raw_path.exists():
        raise FileNotFoundError(f"找不到 RAW 文件: {raw_path}")
    expected_size = safe_num_voxels(shape_zyx) * int(np.dtype(dtype).itemsize)
    actual_size = int(raw_path.stat().st_size)
    if expected_size != actual_size:
        raise ValueError(
            f"RAW 文件大小不匹配！\n理论大小: {expected_size} bytes\n实际大小: {actual_size} bytes\n"
            f"请检查 SHAPE_ZYX 或 DTYPE。"
        )
    print(f"RAW 文件大小校验通过：{actual_size:,} bytes")


def build_positions(shape_zyx, block_size, stride, max_blocks=None):
    z_size, y_size, x_size = shape_zyx

    def axis_starts(axis_size):
        if axis_size < block_size:
            raise ValueError(f"轴尺寸 {axis_size} 小于 BLOCK_SIZE={block_size}")
        starts = list(range(0, axis_size - block_size + 1, stride))
        last = axis_size - block_size
        if INCLUDE_EDGE_BLOCKS and last not in starts:
            starts.append(last)
        return sorted(set(int(v) for v in starts))

    positions = []
    for z in axis_starts(z_size):
        for y in axis_starts(y_size):
            for x in axis_starts(x_size):
                positions.append((z, y, x))
    if max_blocks is not None:
        positions = positions[:int(max_blocks)]
    return positions


def assign_spatial_split(start_value, axis_size=1000, block_size=200):
    max_start = axis_size - block_size
    ratio = start_value / max(max_start, 1)
    if ratio < 0.70:
        return "train"
    elif ratio < 0.85:
        return "val"
    return "test"


def sample_id_from_position(idx, z, y, x):
    return f"{DATASET_PREFIX}_{idx:04d}_z{z}_y{y}_x{x}_b{BLOCK_SIZE}"


def crop_raw_blocks():
    index_csv = SUMMARY_OUT / f"{DATASET_PREFIX}_blocks_index_v2.csv"
    if index_csv.exists() and SKIP_EXISTING_FILES and (not FORCE_REBUILD_DATASET):
        print("Step 1 已存在，直接读取:", index_csv)
        return pd.read_csv(index_csv)
    dtype = resolve_dtype(DTYPE)
    check_raw_file(RAW_PATH, SHAPE_ZYX, dtype)
    volume = np.memmap(RAW_PATH, dtype=dtype, mode="r", shape=SHAPE_ZYX)
    positions = build_positions(SHAPE_ZYX, BLOCK_SIZE, STRIDE, MAX_BLOCKS)
    index_xlsx = SUMMARY_OUT / f"{DATASET_PREFIX}_blocks_index_v2.xlsx"

    print("=" * 90)
    print("Step 1: RAW 裁剪开始")
    print("dtype:", dtype)
    print("总裁剪块数量:", len(positions))
    print("输出目录:", RAW_OUT)
    print("=" * 90)

    rows = []
    for idx, (z, y, x) in enumerate(tqdm(positions, desc="RAW 裁剪")):
        sample_id = sample_id_from_position(idx, z, y, x)
        raw_file = RAW_OUT / f"{sample_id}.raw"
        if (not SKIP_EXISTING_FILES) or FORCE_REBUILD_DATASET or (not raw_file.exists()):
            block = np.asarray(volume[z:z+BLOCK_SIZE, y:y+BLOCK_SIZE, x:x+BLOCK_SIZE])
            block.astype(dtype).tofile(raw_file)
        rows.append({
            "sample_id": sample_id,
            "rock_id": DATASET_PREFIX,
            "rock_name": DATASET_PREFIX,
            "z_start": z, "y_start": y, "x_start": x,
            "z_end": z + BLOCK_SIZE, "y_end": y + BLOCK_SIZE, "x_end": x + BLOCK_SIZE,
            "block_size": BLOCK_SIZE,
            "stride": STRIDE,
            "voxel_size_um": VOXEL_SIZE_UM,
            "raw_dtype": str(dtype),
            "raw_file": str(raw_file),
            "spatial_group_x": int(x // BLOCK_SIZE),
            "spatial_group_y": int(y // BLOCK_SIZE),
            "spatial_group_z": int(z // BLOCK_SIZE),
            "split_by_x": assign_spatial_split(x, SHAPE_ZYX[2], BLOCK_SIZE),
            "split_by_z": assign_spatial_split(z, SHAPE_ZYX[0], BLOCK_SIZE),
        })
        if idx % 10 == 0:
            pd.DataFrame(rows).to_csv(index_csv, index=False, encoding="utf-8-sig")

    df = pd.DataFrame(rows)
    df.to_csv(index_csv, index=False, encoding="utf-8-sig")
    df.to_excel(index_xlsx, index=False)
    print("Step 1 完成:", index_csv)
    return df


def threshold_pore(block: np.ndarray) -> np.ndarray:
    if PORE_MODE.lower() == "equal":
        pore = (block == PORE_THRESHOLD)
    elif PORE_MODE.lower() == "le":
        pore = (block <= PORE_THRESHOLD)
    else:
        raise ValueError("PORE_MODE 只能为 'equal' 或 'le'")
    if MIN_OBJECT_SIZE is not None and MIN_OBJECT_SIZE > 1:
        pore = remove_small_objects(pore.astype(bool), min_size=MIN_OBJECT_SIZE)
    return pore.astype(bool)


def read_raw_block(raw_file: Path):
    dtype = resolve_dtype(DTYPE)
    block = np.fromfile(raw_file, dtype=dtype)
    expected = BLOCK_SIZE ** 3
    if block.size != expected:
        raise ValueError(f"裁剪 raw 文件体素数不匹配: {raw_file}; 实际 {block.size}, 理论 {expected}")
    return block.reshape((BLOCK_SIZE, BLOCK_SIZE, BLOCK_SIZE))


def get_axis_percolating_mask(binary_mask: np.ndarray, axis: int):
    labels, num = ndi.label(binary_mask, structure=np.ones((3, 3, 3), dtype=bool))
    if num == 0:
        return np.zeros_like(binary_mask, dtype=bool), False, 0
    face_min = np.take(labels, indices=0, axis=axis)
    face_max = np.take(labels, indices=labels.shape[axis] - 1, axis=axis)
    labs_min = np.unique(face_min)
    labs_max = np.unique(face_max)
    both = np.intersect1d(labs_min, labs_max)
    both = both[both > 0]
    if both.size == 0:
        return np.zeros_like(binary_mask, dtype=bool), False, 0
    mask = np.isin(labels, both)
    return mask.astype(bool), True, int(both.size)


def segment_pores_directional():
    index_csv = SUMMARY_OUT / f"{DATASET_PREFIX}_blocks_index_v2.csv"
    if not index_csv.exists():
        raise FileNotFoundError("请先运行 Step 1：RAW 裁剪。")
    summary_csv = SUMMARY_OUT / f"{DATASET_PREFIX}_pore_summary_v2.csv"
    summary_xlsx = SUMMARY_OUT / f"{DATASET_PREFIX}_pore_summary_v2.xlsx"
    if summary_csv.exists() and SKIP_EXISTING_FILES and (not FORCE_REBUILD_DATASET):
        print("Step 2 已存在，直接读取:", summary_csv)
        return pd.read_csv(summary_csv)
    index_df = pd.read_csv(index_csv)

    print("=" * 90)
    print("Step 2: 孔隙分割与方向性贯通孔提取开始")
    print("样本数量:", len(index_df))
    print("=" * 90)

    summary_rows = []
    for _, row in tqdm(index_df.iterrows(), total=len(index_df), desc="孔隙分割"):
        sample_id = row["sample_id"]
        raw_file = Path(row["raw_file"])
        pore_file = PORE_OUT / f"{sample_id}.pore.npy"
        connected_files = {d: PORE_OUT / f"{sample_id}.connected_pore_{d}.npy" for d in ["x", "y", "z"]}
        try:
            if SKIP_EXISTING_FILES and (not FORCE_REBUILD_DATASET) and pore_file.exists() and all(p.exists() for p in connected_files.values()):
                pore_mask = np.load(pore_file).astype(bool)
                connected = {d: np.load(connected_files[d]).astype(bool) for d in ["x", "y", "z"]}
                per = {d: bool(connected[d].sum() > 0) for d in ["x", "y", "z"]}
                ncomp = {d: np.nan for d in ["x", "y", "z"]}
            else:
                block = read_raw_block(raw_file)
                pore_mask = threshold_pore(block)
                connected = {}
                per = {}
                ncomp = {}
                for d in ["x", "y", "z"]:
                    connected[d], per[d], ncomp[d] = get_axis_percolating_mask(pore_mask, axis=DIRECTION_AXIS_MASK[d])
                np.save(pore_file, pore_mask.astype(bool))
                for d in ["x", "y", "z"]:
                    np.save(connected_files[d], connected[d].astype(bool))

            porosity = float(np.mean(pore_mask))
            conn_por = {d: float(np.mean(connected[d])) for d in ["x", "y", "z"]}
            status = "ok"
        except Exception as e:
            print(f"[PORE ERROR] {sample_id}: {e}")
            porosity = np.nan
            conn_por = {d: np.nan for d in ["x", "y", "z"]}
            per = {d: False for d in ["x", "y", "z"]}
            ncomp = {d: 0 for d in ["x", "y", "z"]}
            status = f"failed: {e}"

        out = row.to_dict()
        out.update({
            "porosity": porosity,
            "percolates_x": int(per["x"]), "percolates_y": int(per["y"]), "percolates_z": int(per["z"]),
            "num_percolating_components_x": ncomp["x"],
            "num_percolating_components_y": ncomp["y"],
            "num_percolating_components_z": ncomp["z"],
            "connected_porosity_x": conn_por["x"],
            "connected_porosity_y": conn_por["y"],
            "connected_porosity_z": conn_por["z"],
            "connected_porosity": np.nanmax([conn_por["x"], conn_por["y"], conn_por["z"]]),
            "pore_file": str(pore_file),
            "connected_pore_x_file": str(connected_files["x"]),
            "connected_pore_y_file": str(connected_files["y"]),
            "connected_pore_z_file": str(connected_files["z"]),
            "pore_status": status,
        })
        summary_rows.append(out)
        if len(summary_rows) % 10 == 0:
            pd.DataFrame(summary_rows).to_csv(summary_csv, index=False, encoding="utf-8-sig")

    df = pd.DataFrame(summary_rows)
    df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
    df.to_excel(summary_xlsx, index=False)
    print("Step 2 完成:", summary_csv)
    return df


In [ ]:

# ============================================================
# 3. PNM 网络、方向性渗透率与图特征函数
# ============================================================
PORE_COLUMNS = [
    "Pore ID", "Volume [cm^3]", "Area [cm^2]", "EqRadius [cm]", "LabelID", "Pressure",
    "X Coord [cm]", "Y Coord [cm]", "Z Coord [cm]", "Coordination Number",
]
THROAT_COLUMNS = [
    "Throat ID", "Area [cm^2]", "EqRadius [cm]", "ChannelLength [cm]", "FlowRatePerSec [cm^3]",
    "Pore ID #1", "Pore ID #2",
]

def safe_get(obj, keys, default=None):
    for key in keys:
        try:
            if key in obj.keys():
                return np.asarray(obj[key])
        except Exception:
            pass
        try:
            return np.asarray(obj[key])
        except Exception:
            pass
    return default


def get_pore_throat_arrays(pn):
    coords = np.asarray(pn["pore.coords"], dtype=float)
    conns = np.asarray(pn["throat.conns"], dtype=int)
    return coords, conns


def extract_network_by_porespy(mask: np.ndarray):
    if mask.sum() == 0:
        return None, None
    phases = mask.astype(np.int32)
    try:
        snow = ps.networks.snow2(
            phases=phases,
            voxel_size=VOXEL_SIZE_UM,
            boundary_width=SNOW_BOUNDARY_WIDTH,
            sigma=SNOW_SIGMA,
            r_max=SNOW_R_MAX,
        )
    except TypeError:
        snow = ps.networks.snow2(phases=phases, voxel_size=VOXEL_SIZE_UM)
    return snow.network, getattr(snow, "regions", None)


def network_to_openpnm(net):
    try:
        return op.io.network_from_porespy(net)
    except Exception:
        pass
    coords = np.asarray(net["pore.coords"], dtype=float)
    conns = np.asarray(net["throat.conns"], dtype=int)
    try:
        pn = op.network.Network(coords=coords, conns=conns)
    except Exception:
        pn = op.network.GenericNetwork()
        pn["pore.coords"] = coords
        pn["throat.conns"] = conns
    for key, value in net.items():
        try:
            pn[key] = value
        except Exception:
            pass
    return pn


def build_pnm_excel_tables(pn, net, origin_zyx):
    coords_um, conns = get_pore_throat_arrays(pn)
    n_pores = coords_um.shape[0]
    n_throats = conns.shape[0]
    z0, y0, x0 = origin_zyx

    coords_global_um = coords_um.copy()
    coords_global_um[:, 0] += x0 * VOXEL_SIZE_UM
    coords_global_um[:, 1] += y0 * VOXEL_SIZE_UM
    coords_global_um[:, 2] += z0 * VOXEL_SIZE_UM

    pore_diameter_um = safe_get(pn, ["pore.diameter", "pore.equivalent_diameter", "pore.inscribed_diameter"], default=None)
    if pore_diameter_um is None:
        pore_diameter_um = safe_get(net, ["pore.diameter", "pore.equivalent_diameter", "pore.inscribed_diameter"], default=np.ones(n_pores) * VOXEL_SIZE_UM)
    pore_volume_um3 = safe_get(pn, ["pore.volume"], default=None)
    if pore_volume_um3 is None:
        pore_volume_um3 = safe_get(net, ["pore.volume"], default=(4.0 / 3.0) * np.pi * (pore_diameter_um / 2.0) ** 3)
    pore_area_um2 = safe_get(pn, ["pore.surface_area", "pore.area"], default=None)
    if pore_area_um2 is None:
        pore_area_um2 = safe_get(net, ["pore.surface_area", "pore.area"], default=4.0 * np.pi * (pore_diameter_um / 2.0) ** 2)
    pore_label = safe_get(pn, ["pore.region_label", "pore.label"], default=None)
    if pore_label is None:
        pore_label = safe_get(net, ["pore.region_label", "pore.label"], default=np.arange(n_pores))

    coordination = np.zeros(n_pores, dtype=int)
    for p1, p2 in conns:
        if 0 <= p1 < n_pores:
            coordination[p1] += 1
        if 0 <= p2 < n_pores:
            coordination[p2] += 1

    coords_cm = coords_global_um * 1e-4
    pore_volume_cm3 = pore_volume_um3 * 1e-12
    pore_area_cm2 = pore_area_um2 * 1e-8
    pore_eq_radius_cm = (pore_diameter_um / 2.0) * 1e-4

    pore_rows = []
    for i in range(n_pores):
        pore_rows.append([
            i, float(pore_volume_cm3[i]), float(pore_area_cm2[i]), float(pore_eq_radius_cm[i]),
            int(pore_label[i]) if i < len(pore_label) else i,
            130000,
            float(coords_cm[i, 0]), float(coords_cm[i, 1]), float(coords_cm[i, 2]), int(coordination[i]),
        ])
    pore_df = pd.DataFrame(pore_rows, columns=PORE_COLUMNS)

    throat_diameter_um = safe_get(pn, ["throat.diameter", "throat.equivalent_diameter", "throat.inscribed_diameter"], default=None)
    if throat_diameter_um is None:
        throat_diameter_um = safe_get(net, ["throat.diameter", "throat.equivalent_diameter", "throat.inscribed_diameter"], default=np.ones(n_throats) * VOXEL_SIZE_UM)
    throat_area_um2 = safe_get(pn, ["throat.cross_sectional_area", "throat.area"], default=None)
    if throat_area_um2 is None:
        throat_area_um2 = safe_get(net, ["throat.cross_sectional_area", "throat.area"], default=np.pi * (throat_diameter_um / 2.0) ** 2)
    throat_length_um = safe_get(pn, ["throat.total_length", "throat.direct_length", "throat.length"], default=None)
    if throat_length_um is None:
        throat_length_um = safe_get(net, ["throat.total_length", "throat.direct_length", "throat.length"], default=None)
    if throat_length_um is None:
        c1 = coords_um[conns[:, 0]]
        c2 = coords_um[conns[:, 1]]
        throat_length_um = np.linalg.norm(c1 - c2, axis=1)

    throat_area_cm2 = throat_area_um2 * 1e-8
    throat_eq_radius_cm = (throat_diameter_um / 2.0) * 1e-4
    throat_length_cm = throat_length_um * 1e-4
    throat_rows = []
    for i in range(n_throats):
        throat_rows.append([
            i, float(throat_area_cm2[i]), float(throat_eq_radius_cm[i]), float(throat_length_cm[i]),
            0.0, int(conns[i, 0]), int(conns[i, 1]),
        ])
    throat_df = pd.DataFrame(throat_rows, columns=THROAT_COLUMNS)
    return pore_df, throat_df


def write_pnm_xlsx(pore_df, throat_df, out_path: Path):
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        pore_df.to_excel(writer, sheet_name="Pores", index=False)
        throat_df.to_excel(writer, sheet_name="Throats", index=False)
        pd.DataFrame().to_excel(writer, sheet_name="Sheet3", index=False, header=False)


def compute_hydraulic_conductance(pn):
    coords_um, conns = get_pore_throat_arrays(pn)
    n_throats = conns.shape[0]
    throat_diameter_um = safe_get(pn, ["throat.diameter", "throat.equivalent_diameter", "throat.inscribed_diameter"], default=None)
    if throat_diameter_um is None:
        throat_area_um2 = safe_get(pn, ["throat.cross_sectional_area", "throat.area"], default=None)
        if throat_area_um2 is not None:
            throat_diameter_um = 2.0 * np.sqrt(throat_area_um2 / np.pi)
        else:
            throat_diameter_um = np.ones(n_throats) * VOXEL_SIZE_UM
    throat_length_um = safe_get(pn, ["throat.total_length", "throat.direct_length", "throat.length"], default=None)
    if throat_length_um is None:
        c1 = coords_um[conns[:, 0]]
        c2 = coords_um[conns[:, 1]]
        throat_length_um = np.linalg.norm(c1 - c2, axis=1)
    throat_length_um = np.maximum(throat_length_um, VOXEL_SIZE_UM)
    r_um = np.maximum(throat_diameter_um / 2.0, VOXEL_SIZE_UM / 10.0)
    g = np.pi * r_um ** 4 / (8.0 * VISCOSITY * throat_length_um)
    return g


def get_boundary_pores_from_coords(pn, axis_index, side="min"):
    coords = np.asarray(pn["pore.coords"], dtype=float)
    vals = coords[:, axis_index]
    block_len_um = BLOCK_SIZE * VOXEL_SIZE_UM
    tol = 2.0 * VOXEL_SIZE_UM
    if side == "min":
        pores = np.where(vals <= tol)[0]
    else:
        pores = np.where(vals >= block_len_um - tol)[0]
    if len(pores) == 0:
        vmin = np.min(vals)
        vmax = np.max(vals)
        if side == "min":
            pores = np.where(vals <= vmin + tol)[0]
        else:
            pores = np.where(vals >= vmax - tol)[0]
    return pores


def calculate_absolute_permeability_directional(pn, direction):
    diag = {"n_pores_used": 0, "n_throats_used": 0, "n_inlet_pores": 0, "n_outlet_pores": 0, "flow_rate": np.nan}
    if pn is None or "throat.conns" not in pn.keys() or "pore.coords" not in pn.keys():
        return np.nan, "invalid_network", diag
    conns = np.asarray(pn["throat.conns"], dtype=np.int64)
    coords = np.asarray(pn["pore.coords"], dtype=float)
    if conns.size == 0 or coords.size == 0:
        return np.nan, "empty_network", diag
    n_pores = coords.shape[0]
    axis_index = DIRECTION_AXIS_PN[direction]
    inlets = np.asarray(get_boundary_pores_from_coords(pn, axis_index, side="min"), dtype=np.int64)
    outlets = np.asarray(get_boundary_pores_from_coords(pn, axis_index, side="max"), dtype=np.int64)
    inlets = inlets[(inlets >= 0) & (inlets < n_pores)]
    outlets = outlets[(outlets >= 0) & (outlets < n_pores)]
    diag["n_inlet_pores"] = int(len(inlets))
    diag["n_outlet_pores"] = int(len(outlets))
    if len(inlets) == 0 or len(outlets) == 0:
        return np.nan, "boundary_pores_missing", diag
    try:
        g_all = np.asarray(compute_hydraulic_conductance(pn), dtype=np.float64)
    except Exception as e:
        return np.nan, f"conductance_failed: {e}", diag
    valid_edges = ((conns[:, 0] >= 0) & (conns[:, 0] < n_pores) & (conns[:, 1] >= 0) & (conns[:, 1] < n_pores) & np.isfinite(g_all) & (g_all > 0))
    conns = conns[valid_edges]
    g_all = g_all[valid_edges]
    if conns.shape[0] == 0:
        return np.nan, "no_valid_throats", diag
    try:
        G = nx.Graph()
        G.add_nodes_from(range(n_pores))
        G.add_edges_from((int(a), int(b)) for a, b in conns)
        inlet_set = set(int(i) for i in inlets)
        outlet_set = set(int(o) for o in outlets)
        active_nodes = set()
        for comp in nx.connected_components(G):
            if comp & inlet_set and comp & outlet_set:
                active_nodes.update(comp)
        if len(active_nodes) == 0:
            return np.nan, "network_not_connected_between_boundaries", diag
        active_nodes = np.array(sorted(active_nodes), dtype=np.int64)
    except Exception as e:
        return np.nan, f"connectivity_check_failed: {e}", diag
    old_to_new = -np.ones(n_pores, dtype=np.int64)
    old_to_new[active_nodes] = np.arange(len(active_nodes), dtype=np.int64)
    edge_mask = (old_to_new[conns[:, 0]] >= 0) & (old_to_new[conns[:, 1]] >= 0)
    conns_active_old = conns[edge_mask]
    g = g_all[edge_mask]
    if conns_active_old.shape[0] == 0:
        return np.nan, "no_active_throats", diag
    conns_active = np.column_stack([old_to_new[conns_active_old[:, 0]], old_to_new[conns_active_old[:, 1]]]).astype(np.int64)
    n = len(active_nodes)
    diag["n_pores_used"] = int(n)
    diag["n_throats_used"] = int(conns_active.shape[0])
    inlet_active = np.isin(active_nodes, inlets)
    outlet_active = np.isin(active_nodes, outlets)
    fixed = inlet_active | outlet_active
    if inlet_active.sum() == 0 or outlet_active.sum() == 0:
        return np.nan, "active_boundary_pores_missing", diag
    p_fixed = np.zeros(n, dtype=np.float64)
    p_fixed[inlet_active] = PIN
    p_fixed[outlet_active] = POUT
    rows, cols, data = [], [], []
    for (i, j), gij in zip(conns_active, g):
        i = int(i); j = int(j); gij = float(gij)
        rows.extend([i, i, j, j]); cols.extend([i, j, i, j]); data.extend([gij, -gij, -gij, gij])
    Lmat = sparse.csr_matrix((data, (rows, cols)), shape=(n, n))
    free = ~fixed
    p = p_fixed.copy()
    try:
        if np.any(free):
            A_ff = Lmat[free][:, free]
            b_f = -Lmat[free][:, fixed].dot(p_fixed[fixed])
            try:
                p_free, info = cg(A_ff, b_f, rtol=1e-8, maxiter=3000)
                if info != 0 or not np.all(np.isfinite(p_free)):
                    p_free = spsolve(A_ff, b_f)
            except Exception:
                p_free = spsolve(A_ff, b_f)
            if not np.all(np.isfinite(p_free)):
                return np.nan, "pressure_solution_nonfinite", diag
            p[free] = p_free
    except Exception as e:
        return np.nan, f"linear_solve_failed: {e}", diag
    Q = 0.0
    for (i, j), gij in zip(conns_active, g):
        i = int(i); j = int(j); gij = float(gij)
        if inlet_active[i] and not inlet_active[j]:
            Q += gij * max(p[i] - p[j], 0.0)
        elif inlet_active[j] and not inlet_active[i]:
            Q += gij * max(p[j] - p[i], 0.0)
    diag["flow_rate"] = float(Q)
    if Q <= 0 or not np.isfinite(Q):
        return np.nan, "zero_or_nonfinite_flow", diag
    L = np.max(coords[active_nodes, axis_index]) - np.min(coords[active_nodes, axis_index])
    L = max(float(L), VOXEL_SIZE_UM)
    side_um = BLOCK_SIZE * VOXEL_SIZE_UM
    A = side_um * side_um
    k_um2 = Q * VISCOSITY * L / (A * (PIN - POUT))
    if not np.isfinite(k_um2) or k_um2 <= 0:
        return np.nan, "invalid_permeability", diag
    return float(k_um2), "ok", diag


def graph_features_from_pnm_tables(pore_df, throat_df):
    features = {"num_pores": len(pore_df), "num_throats": len(throat_df)}
    n_pores = len(pore_df)
    n_throats = len(throat_df)
    if n_pores == 0:
        features.update({
            "avg_pore_radius_cm": np.nan, "std_pore_radius_cm": np.nan,
            "avg_throat_radius_cm": np.nan, "std_throat_radius_cm": np.nan,
            "avg_throat_length_cm": np.nan, "std_throat_length_cm": np.nan,
            "avg_coordination_number": np.nan, "max_coordination_number": np.nan,
            "network_density": np.nan, "avg_betweenness": np.nan, "max_betweenness": np.nan,
            "avg_closeness": np.nan, "avg_pagerank": np.nan, "max_pagerank": np.nan,
        })
        return features
    pore_radius = pd.to_numeric(pore_df["EqRadius [cm]"], errors="coerce")
    coord_num = pd.to_numeric(pore_df["Coordination Number"], errors="coerce")
    features["avg_pore_radius_cm"] = float(pore_radius.mean())
    features["std_pore_radius_cm"] = float(pore_radius.std())
    features["avg_coordination_number"] = float(coord_num.mean())
    features["max_coordination_number"] = float(coord_num.max())
    if n_throats > 0:
        throat_radius = pd.to_numeric(throat_df["EqRadius [cm]"], errors="coerce")
        throat_length = pd.to_numeric(throat_df["ChannelLength [cm]"], errors="coerce")
        features["avg_throat_radius_cm"] = float(throat_radius.mean())
        features["std_throat_radius_cm"] = float(throat_radius.std())
        features["avg_throat_length_cm"] = float(throat_length.mean())
        features["std_throat_length_cm"] = float(throat_length.std())
    else:
        features["avg_throat_radius_cm"] = np.nan
        features["std_throat_radius_cm"] = np.nan
        features["avg_throat_length_cm"] = np.nan
        features["std_throat_length_cm"] = np.nan
    try:
        G = nx.Graph()
        G.add_nodes_from(range(n_pores))
        if n_throats > 0:
            edges = []
            for _, r in throat_df.iterrows():
                p1 = int(r["Pore ID #1"]); p2 = int(r["Pore ID #2"])
                length = float(r["ChannelLength [cm]"]); radius = float(r["EqRadius [cm]"])
                if p1 != p2 and 0 <= p1 < n_pores and 0 <= p2 < n_pores:
                    edges.append((p1, p2, {"length": max(length, 1e-30), "radius": radius}))
            G.add_edges_from(edges)
        features["network_density"] = float(nx.density(G))
        if G.number_of_edges() > 0 and G.number_of_nodes() > 1:
            if G.number_of_nodes() <= CENTRALITY_MAX_NODES_EXACT:
                bet = nx.betweenness_centrality(G, weight="length", normalized=True)
            else:
                k = min(BETWEENNESS_APPROX_K, G.number_of_nodes())
                bet = nx.betweenness_centrality(G, k=k, weight="length", normalized=True, seed=SEED)
            clo = nx.closeness_centrality(G, distance="length")
            pr = nx.pagerank(G, weight="radius")
            features["avg_betweenness"] = float(np.mean(list(bet.values())))
            features["max_betweenness"] = float(np.max(list(bet.values())))
            features["avg_closeness"] = float(np.mean(list(clo.values())))
            features["avg_pagerank"] = float(np.mean(list(pr.values())))
            features["max_pagerank"] = float(np.max(list(pr.values())))
        else:
            features["avg_betweenness"] = 0.0
            features["max_betweenness"] = 0.0
            features["avg_closeness"] = 0.0
            features["avg_pagerank"] = 0.0
            features["max_pagerank"] = 0.0
    except Exception as e:
        features["network_density"] = np.nan
        features["avg_betweenness"] = np.nan
        features["max_betweenness"] = np.nan
        features["avg_closeness"] = np.nan
        features["avg_pagerank"] = np.nan
        features["max_pagerank"] = np.nan
        features["graph_feature_error"] = str(e)
    return features


def write_directional_permeability_xlsx(sample_id, row_result, out_path: Path):
    rows = []
    for d in ["x", "y", "z"]:
        rows.append({
            "sample_id": sample_id,
            "direction": d.upper(),
            "percolates": int(row_result.get(f"percolates_{d}", 0)),
            "permeability_um2": float(row_result.get(f"k{d}_um2", 0.0)) if pd.notna(row_result.get(f"k{d}_um2", np.nan)) else np.nan,
            "status": row_result.get(f"k{d}_status", ""),
            "pnm_xlsx": row_result.get(f"pnm_{d}_xlsx", ""),
        })
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        pd.DataFrame(rows).to_excel(writer, sheet_name="Directional_Permeability", index=False)


def process_permeation_row(row_dict):
    row = pd.Series(row_dict)
    sample_id = row["sample_id"]
    origin_zyx = (int(row.get("z_start", 0)), int(row.get("y_start", 0)), int(row.get("x_start", 0)))
    permeability_xlsx = PERM_OUT / f"{sample_id}.Directional_KTensor_v2.xlsx"
    result = {
        "sample_id": sample_id,
        "z_start": row.get("z_start", np.nan), "y_start": row.get("y_start", np.nan), "x_start": row.get("x_start", np.nan),
        "block_size": row.get("block_size", BLOCK_SIZE),
        "percolates_x": int(row.get("percolates_x", 0)),
        "percolates_y": int(row.get("percolates_y", 0)),
        "percolates_z": int(row.get("percolates_z", 0)),
        "kx_um2": np.nan, "ky_um2": np.nan, "kz_um2": np.nan,
        "kx_status": "not_percolating", "ky_status": "not_percolating", "kz_status": "not_percolating",
        "permeability_xlsx": str(permeability_xlsx),
        "permeation_status": "ok",
    }
    graph_feature_rows = []
    try:
        for direction in ["x", "y", "z"]:
            per_flag = int(row.get(f"percolates_{direction}", 0))
            connected_file = Path(row[f"connected_pore_{direction}_file"])
            pnm_xlsx = PORE_OUT / f"{sample_id}.PNM_{direction}.flooded.am.xlsx"
            result[f"pnm_{direction}_xlsx"] = str(pnm_xlsx)

            if per_flag == 0:
                if (not pnm_xlsx.exists()) or (not SKIP_EXISTING_FILES) or FORCE_REBUILD_DATASET:
                    write_pnm_xlsx(pd.DataFrame(columns=PORE_COLUMNS), pd.DataFrame(columns=THROAT_COLUMNS), pnm_xlsx)
                result[f"k{direction}_um2"] = 0.0
                result[f"k{direction}_status"] = "not_percolating"
                continue

            if SKIP_EXISTING_FILES and (not FORCE_REBUILD_DATASET) and pnm_xlsx.exists() and (not pd.isna(row.get(f"k{direction}_um2", np.nan))):
                # 仅用于重复运行时兜底；通常当前 row 不含 k 值，因此会正常重算。
                result[f"k{direction}_um2"] = row.get(f"k{direction}_um2")
                result[f"k{direction}_status"] = row.get(f"k{direction}_status", "exists")
                try:
                    pore_df = pd.read_excel(pnm_xlsx, sheet_name="Pores")
                    throat_df = pd.read_excel(pnm_xlsx, sheet_name="Throats")
                    gf = graph_features_from_pnm_tables(pore_df, throat_df)
                    graph_feature_rows.append({f"{direction}_{k}": v for k, v in gf.items()})
                except Exception:
                    pass
                continue

            mask = np.load(connected_file).astype(bool)
            if mask.sum() == 0:
                write_pnm_xlsx(pd.DataFrame(columns=PORE_COLUMNS), pd.DataFrame(columns=THROAT_COLUMNS), pnm_xlsx)
                result[f"k{direction}_um2"] = np.nan
                result[f"k{direction}_status"] = "empty_mask"
                continue

            net, regions = extract_network_by_porespy(mask)
            pn = network_to_openpnm(net)
            pore_df, throat_df = build_pnm_excel_tables(pn, net, origin_zyx=origin_zyx)
            write_pnm_xlsx(pore_df, throat_df, pnm_xlsx)
            k_val, k_status, k_diag = calculate_absolute_permeability_directional(pn, direction)
            result[f"k{direction}_um2"] = k_val
            result[f"k{direction}_status"] = k_status
            for key, val in k_diag.items():
                result[f"k{direction}_{key}"] = val
            gf = graph_features_from_pnm_tables(pore_df, throat_df)
            graph_feature_rows.append({f"{direction}_{k}": v for k, v in gf.items()})

        for gf in graph_feature_rows:
            result.update(gf)
        for direction in ["x", "y", "z"]:
            if int(row.get(f"percolates_{direction}", 0)) == 1:
                for key in [
                    "num_pores", "num_throats", "avg_pore_radius_cm", "std_pore_radius_cm",
                    "avg_throat_radius_cm", "std_throat_radius_cm", "avg_throat_length_cm", "std_throat_length_cm",
                    "avg_coordination_number", "max_coordination_number", "network_density", "avg_betweenness", "max_betweenness",
                    "avg_closeness", "avg_pagerank", "max_pagerank",
                ]:
                    dkey = f"{direction}_{key}"
                    if dkey in result:
                        result[key] = result[dkey]
                break
        if SAVE_INDIVIDUAL_PERMEABILITY_XLSX:
            write_directional_permeability_xlsx(sample_id, result, permeability_xlsx)
    except Exception as e:
        result["permeation_status"] = f"failed: {e}"
        try:
            if SAVE_INDIVIDUAL_PERMEABILITY_XLSX:
                write_directional_permeability_xlsx(sample_id, result, permeability_xlsx)
        except Exception:
            pass
    return result


def compute_directional_permeability_dataset():
    pore_csv = SUMMARY_OUT / f"{DATASET_PREFIX}_pore_summary_v2.csv"
    if not pore_csv.exists():
        raise FileNotFoundError("请先运行 Step 2：孔隙分割。")
    summary_csv = SUMMARY_OUT / f"{DATASET_PREFIX}_permeation_summary_v2.csv"
    summary_xlsx = SUMMARY_OUT / f"{DATASET_PREFIX}_permeation_summary_v2.xlsx"
    if summary_csv.exists() and SKIP_EXISTING_FILES and (not FORCE_REBUILD_DATASET):
        print("Step 3 已存在，直接读取:", summary_csv)
        return pd.read_csv(summary_csv)
    df = pd.read_csv(pore_csv)

    print("=" * 90)
    print("Step 3: 方向性 PNM 网络提取、渗透率计算与图特征提取开始")
    print("样本数量:", len(df))
    print("是否并行:", PARALLEL_PERMEATION, "任务数:", N_JOBS_PERMEATION)
    print("=" * 90)

    rows = df.to_dict("records")
    if PARALLEL_PERMEATION and len(rows) > 1:
        with tqdm_joblib(tqdm(total=len(rows), desc="方向性渗透率计算", unit="样本")):
            summary_rows = Parallel(n_jobs=N_JOBS_PERMEATION, backend="loky", batch_size=1, verbose=0)(
                delayed(process_permeation_row)(r) for r in rows
            )
    else:
        summary_rows = [process_permeation_row(r) for r in tqdm(rows, desc="方向性渗透率计算", unit="样本")]

    out_df = pd.DataFrame(summary_rows)
    out_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
    out_df.to_excel(summary_xlsx, index=False)
    out_df.to_csv(PERM_OUT / f"{DATASET_PREFIX}_permeation_summary_v2.csv", index=False, encoding="utf-8-sig")
    out_df.to_excel(PERM_OUT / f"{DATASET_PREFIX}_permeation_summary_v2.xlsx", index=False)
    print("Step 3 完成:", summary_csv)
    return out_df


In [ ]:

# ============================================================
# 4. 力学代理参数、固体连通与骨架特征函数
# ============================================================

def compute_pore_surface_area_density(pore_mask):
    face_area = {0: VX_UM * VY_UM, 1: VX_UM * VZ_UM, 2: VY_UM * VZ_UM}
    area_um2 = 0.0
    for axis in [0, 1, 2]:
        sl1 = [slice(None)] * 3
        sl2 = [slice(None)] * 3
        sl1[axis] = slice(0, -1)
        sl2[axis] = slice(1, None)
        interface = pore_mask[tuple(sl1)] != pore_mask[tuple(sl2)]
        area_um2 += np.sum(interface) * face_area[axis]
    volume_um3 = pore_mask.shape[0] * VZ_UM * pore_mask.shape[1] * VY_UM * pore_mask.shape[2] * VX_UM
    return float(area_um2), float(area_um2 / max(volume_um3, 1e-30))


def downsample_binary_mask(binary_mask, max_size=100, threshold=0.5):
    max_dim = max(binary_mask.shape)
    if max_dim <= max_size:
        return binary_mask.astype(bool), 1
    factor = int(math.ceil(max_dim / max_size))
    new_shape = tuple((s // factor) * factor for s in binary_mask.shape)
    cropped = binary_mask[0:new_shape[0], 0:new_shape[1], 0:new_shape[2]]
    reduced = block_reduce(cropped.astype(np.float32), block_size=(factor, factor, factor), func=np.mean)
    return (reduced >= threshold).astype(bool), factor


def downsample_solid_mask_for_mechanics(solid_mask, max_size=64, threshold=None):
    if threshold is None:
        threshold = MECH_SOLID_THRESHOLD
    return downsample_binary_mask(solid_mask, max_size=max_size, threshold=threshold)


def _solve_scalar_spring_stiffness_once(solid_mask, axis, max_size, threshold):
    result = {"E": np.nan, "status": "unknown", "n_nodes": 0, "n_edges": 0, "n_inlet_nodes": 0, "n_outlet_nodes": 0, "reaction": np.nan, "max_size_used": max_size, "threshold_used": threshold}
    try:
        solid_ds, factor = downsample_solid_mask_for_mechanics(solid_mask, max_size=max_size, threshold=threshold)
    except Exception as e:
        result["status"] = f"downsample_failed: {e}"
        return result
    solid_perc, is_perc, _ = get_axis_percolating_mask(solid_ds, axis=axis)
    if not is_perc or solid_perc.sum() == 0:
        result["E"] = 0.0
        result["status"] = "not_percolating"
        return result
    coords = np.argwhere(solid_perc)
    n_nodes = coords.shape[0]
    result["n_nodes"] = int(n_nodes)
    if n_nodes < 2:
        result["status"] = "insufficient_nodes"
        return result
    shape = solid_perc.shape
    id_map = -np.ones(shape, dtype=np.int32)
    id_map[solid_perc] = np.arange(n_nodes, dtype=np.int32)
    inlet = coords[:, axis] == 0
    outlet = coords[:, axis] == (shape[axis] - 1)
    result["n_inlet_nodes"] = int(inlet.sum())
    result["n_outlet_nodes"] = int(outlet.sum())
    if inlet.sum() == 0 or outlet.sum() == 0:
        result["status"] = "boundary_nodes_missing"
        return result
    neighbor_shifts = [(1, 0, 0), (0, 1, 0), (0, 0, 1)]
    edge_i, edge_j = [], []
    for i, (z, y, x) in enumerate(coords):
        for dz, dy, dx in neighbor_shifts:
            nz, ny, nx = z + dz, y + dy, x + dx
            if 0 <= nz < shape[0] and 0 <= ny < shape[1] and 0 <= nx < shape[2]:
                j = id_map[nz, ny, nx]
                if j >= 0:
                    edge_i.append(i)
                    edge_j.append(int(j))
    n_edges = len(edge_i)
    result["n_edges"] = int(n_edges)
    if n_edges == 0:
        result["status"] = "no_spring_edges"
        return result
    edge_i = np.asarray(edge_i, dtype=np.int64)
    edge_j = np.asarray(edge_j, dtype=np.int64)
    edge_g = np.ones(n_edges, dtype=np.float64)
    rows = np.concatenate([edge_i, edge_i, edge_j, edge_j])
    cols = np.concatenate([edge_i, edge_j, edge_i, edge_j])
    data = np.concatenate([edge_g, -edge_g, -edge_g, edge_g])
    Lmat = sparse.csr_matrix((data, (rows, cols)), shape=(n_nodes, n_nodes))
    fixed = inlet | outlet
    u = np.zeros(n_nodes, dtype=np.float64)
    u[inlet] = 1.0
    u[outlet] = 0.0
    free = ~fixed
    try:
        if np.any(free):
            A_ff = Lmat[free][:, free]
            b_f = -Lmat[free][:, fixed].dot(u[fixed])
            try:
                u_free, info = cg(A_ff, b_f, rtol=1e-8, maxiter=3000)
                if info != 0 or not np.all(np.isfinite(u_free)):
                    u_free = spsolve(A_ff, b_f)
            except Exception:
                u_free = spsolve(A_ff, b_f)
            if not np.all(np.isfinite(u_free)):
                result["status"] = "displacement_solution_nonfinite"
                return result
            u[free] = u_free
    except Exception as e:
        result["status"] = f"linear_solve_failed: {e}"
        return result
    reaction = 0.0
    for i, j in zip(edge_i, edge_j):
        i = int(i); j = int(j)
        if inlet[i] and not inlet[j]:
            reaction += max(u[i] - u[j], 0.0)
        elif inlet[j] and not inlet[i]:
            reaction += max(u[j] - u[i], 0.0)
    result["reaction"] = float(reaction)
    if reaction <= 0 or not np.isfinite(reaction):
        result["status"] = "zero_or_nonfinite_reaction"
        return result
    L = max(shape[axis] - 1, 1)
    other_axes = [0, 1, 2]
    other_axes.remove(axis)
    area = shape[other_axes[0]] * shape[other_axes[1]]
    E_eff = MATRIX_E_GPA * (reaction * L / max(area, 1e-30))
    if not np.isfinite(E_eff) or E_eff <= 0:
        result["status"] = "invalid_E"
        return result
    result["E"] = float(E_eff)
    result["status"] = "ok"
    return result


def solve_scalar_spring_stiffness(solid_mask, axis):
    max_dim = max(solid_mask.shape)
    attempts = [("downsampled", MECH_MAX_SIZE, MECH_SOLID_THRESHOLD)]
    if max_dim > MECH_MAX_SIZE:
        attempts.append(("full_resolution", max_dim, MECH_SOLID_THRESHOLD))
    last = None
    for attempt_name, max_size, threshold in attempts:
        res = _solve_scalar_spring_stiffness_once(solid_mask, axis=axis, max_size=max_size, threshold=threshold)
        res["attempt"] = attempt_name
        last = res
        if res["status"] == "ok":
            return res
        if attempt_name == "full_resolution" and res["status"] == "not_percolating":
            return res
    if last is None:
        return {"E": np.nan, "status": "no_attempt", "attempt": ""}
    if last.get("status") != "not_percolating":
        last["E"] = np.nan
    return last


def compute_solid_directional_features(solid_mask):
    result = {}
    for direction in ["x", "y", "z"]:
        axis = DIRECTION_AXIS_MASK[direction]
        solid_perc, is_perc, ncomp = get_axis_percolating_mask(solid_mask, axis=axis)
        result[f"solid_percolates_{direction}"] = int(is_perc)
        result[f"solid_percolating_components_{direction}"] = int(ncomp)
        result[f"solid_connected_fraction_{direction}"] = float(np.mean(solid_perc))
    return result


def build_3d_neighbor_kernel():
    kernel = np.ones((3, 3, 3), dtype=np.uint8)
    kernel[1, 1, 1] = 0
    return kernel


def compute_skeleton_directional_features(skeleton_mask):
    result = {}
    for direction in ["x", "y", "z"]:
        axis = DIRECTION_AXIS_MASK[direction]
        skel_perc, is_perc, ncomp = get_axis_percolating_mask(skeleton_mask, axis=axis)
        result[f"solid_skeleton_percolates_{direction}"] = int(is_perc)
        result[f"solid_skeleton_percolating_components_{direction}"] = int(ncomp)
        result[f"solid_skeleton_connected_fraction_{direction}"] = float(np.mean(skel_perc))
    return result


def compute_solid_skeleton_features(solid_mask):
    result = {
        "solid_skeleton_enabled": int(ENABLE_SOLID_SKELETON),
        "solid_skeleton_voxel_count": np.nan,
        "solid_skeleton_volume_fraction": np.nan,
        "solid_skeleton_length_um_approx": np.nan,
        "solid_skeleton_component_count": np.nan,
        "solid_skeleton_endpoint_count": np.nan,
        "solid_skeleton_junction_count": np.nan,
        "solid_skeleton_branch_voxel_count": np.nan,
        "solid_skeleton_avg_degree": np.nan,
        "solid_skeleton_max_degree": np.nan,
        "solid_skeleton_downsample_factor": np.nan,
        "solid_skeleton_status": "disabled",
    }
    for direction in ["x", "y", "z"]:
        result[f"solid_skeleton_percolates_{direction}"] = np.nan
        result[f"solid_skeleton_percolating_components_{direction}"] = np.nan
        result[f"solid_skeleton_connected_fraction_{direction}"] = np.nan
    if not ENABLE_SOLID_SKELETON:
        return result
    try:
        solid_for_skeleton, factor = downsample_binary_mask(solid_mask, max_size=SKELETON_MAX_SIZE, threshold=MECH_SOLID_THRESHOLD)
        if solid_for_skeleton.sum() == 0:
            result["solid_skeleton_status"] = "empty_solid"
            result["solid_skeleton_downsample_factor"] = factor
            return result
        skeleton = skeletonize_3d_compatible(solid_for_skeleton).astype(bool)
        voxel_count = int(skeleton.sum())
        result["solid_skeleton_voxel_count"] = voxel_count
        result["solid_skeleton_volume_fraction"] = float(np.mean(skeleton))
        result["solid_skeleton_downsample_factor"] = int(factor)
        result["solid_skeleton_length_um_approx"] = float(voxel_count * VOXEL_SIZE_UM * factor)
        if voxel_count == 0:
            result["solid_skeleton_status"] = "empty_skeleton"
            return result
        labels, ncomp = ndi.label(skeleton, structure=np.ones((3, 3, 3), dtype=bool))
        result["solid_skeleton_component_count"] = int(ncomp)
        degree_map = ndi.convolve(skeleton.astype(np.uint8), build_3d_neighbor_kernel(), mode="constant", cval=0)
        degree_values = degree_map[skeleton].astype(float)
        result["solid_skeleton_endpoint_count"] = int(np.sum(degree_values == 1))
        result["solid_skeleton_junction_count"] = int(np.sum(degree_values >= 3))
        result["solid_skeleton_branch_voxel_count"] = int(np.sum(degree_values == 2))
        result["solid_skeleton_avg_degree"] = float(np.mean(degree_values))
        result["solid_skeleton_max_degree"] = float(np.max(degree_values))
        result.update(compute_skeleton_directional_features(skeleton))
        result["solid_skeleton_status"] = "ok"
    except Exception as e:
        result["solid_skeleton_status"] = f"failed: {e}"
    return result


def compute_mechanics_features_v2(pore_mask, sample_id, origin_zyx, solid_mask=None):
    porosity = float(np.mean(pore_mask))
    if solid_mask is None:
        solid_mask = ~pore_mask
        solid_threshold_method = "solid_mask_from_inverse_pore_mask"
    else:
        solid_mask = solid_mask.astype(bool)
        solid_threshold_method = "solid_mask_from_non_pore_voxels"
    solid_fraction = float(np.mean(solid_mask))
    pore_area_um2, pore_area_density = compute_pore_surface_area_density(pore_mask)
    solid_features = compute_solid_directional_features(solid_mask)
    skeleton_features = compute_solid_skeleton_features(solid_mask)
    Ez_res = solve_scalar_spring_stiffness(solid_mask, axis=0)
    Ey_res = solve_scalar_spring_stiffness(solid_mask, axis=1)
    Ex_res = solve_scalar_spring_stiffness(solid_mask, axis=2)
    Ex = float(Ex_res.get("E", np.nan)); Ey = float(Ey_res.get("E", np.nan)); Ez = float(Ez_res.get("E", np.nan))
    E_values = np.array([Ex, Ey, Ez], dtype=np.float64)
    finite_positive_E = E_values[np.isfinite(E_values) & (E_values > 0)]
    E_mean = float(np.mean(finite_positive_E)) if finite_positive_E.size > 0 else np.nan
    if np.all(np.isfinite(E_values)) and np.all(E_values > 0):
        E_geom = float((Ex * Ey * Ez) ** (1.0 / 3.0))
        E_anisotropy = float(np.max(E_values) / max(np.min(E_values), 1e-30))
    else:
        E_geom = np.nan
        E_anisotropy = np.nan
    nu_eff = float(np.clip(MATRIX_NU * solid_fraction, 0.01, 0.45))
    if np.isfinite(E_geom) and E_geom > 0:
        K_eff = E_geom / (3.0 * (1.0 - 2.0 * nu_eff))
        G_eff = E_geom / (2.0 * (1.0 + nu_eff))
    else:
        K_eff = np.nan; G_eff = np.nan
    z0, y0, x0 = origin_zyx
    row = {
        "sample_id": sample_id,
        "z_start": z0, "y_start": y0, "x_start": x0,
        "block_size": BLOCK_SIZE,
        "voxel_size_um": VOXEL_SIZE_UM,
        "pore_threshold": PORE_THRESHOLD,
        "pore_threshold_mode": PORE_MODE,
        "solid_threshold_method": solid_threshold_method,
        "porosity": porosity,
        "solid_fraction": solid_fraction,
        "pore_surface_area_um2": pore_area_um2,
        "pore_surface_area_density_um2_per_um3": pore_area_density,
        "matrix_E_GPa": MATRIX_E_GPA,
        "matrix_nu": MATRIX_NU,
        "Ex_spring_GPa": Ex,
        "Ey_spring_GPa": Ey,
        "Ez_spring_GPa": Ez,
        "Ex_status": Ex_res.get("status", ""), "Ey_status": Ey_res.get("status", ""), "Ez_status": Ez_res.get("status", ""),
        "Ex_solver_attempt": Ex_res.get("attempt", ""), "Ey_solver_attempt": Ey_res.get("attempt", ""), "Ez_solver_attempt": Ez_res.get("attempt", ""),
        "Ex_reaction": Ex_res.get("reaction", np.nan), "Ey_reaction": Ey_res.get("reaction", np.nan), "Ez_reaction": Ez_res.get("reaction", np.nan),
        "Ex_n_nodes": Ex_res.get("n_nodes", np.nan), "Ey_n_nodes": Ey_res.get("n_nodes", np.nan), "Ez_n_nodes": Ez_res.get("n_nodes", np.nan),
        "Ex_n_edges": Ex_res.get("n_edges", np.nan), "Ey_n_edges": Ey_res.get("n_edges", np.nan), "Ez_n_edges": Ez_res.get("n_edges", np.nan),
        "E_mean_GPa": E_mean,
        "E_geom_mean_GPa": E_geom,
        "E_anisotropy_max_min": E_anisotropy,
        "nu_eff_estimated": nu_eff,
        "K_eff_estimated_GPa": K_eff,
        "G_eff_estimated_GPa": G_eff,
        "mechanics_method": f"solid_non_pore_downsampled_scalar_spring_network_max_size_{MECH_MAX_SIZE}_skeleton_enabled_{ENABLE_SOLID_SKELETON}",
    }
    row.update(solid_features)
    row.update(skeleton_features)
    return row


def write_mechanics_xlsx(mech_row, out_path: Path):
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        pd.DataFrame([mech_row]).to_excel(writer, sheet_name="Mechanics", index=False)


def process_mechanics_row(row_dict):
    row = pd.Series(row_dict)
    sample_id = row["sample_id"]
    origin_zyx = (int(row.get("z_start", 0)), int(row.get("y_start", 0)), int(row.get("x_start", 0)))
    pore_file = Path(row["pore_file"])
    mech_xlsx = MECH_OUT / f"{sample_id}.Mechanics_v2.xlsx"
    try:
        pore_mask = np.load(pore_file).astype(bool)
        raw_file = Path(row.get("raw_file", ""))
        solid_mask = None
        if raw_file.exists():
            raw_block = read_raw_block(raw_file)
            solid_mask = ~threshold_pore(raw_block)
        mech_row = compute_mechanics_features_v2(pore_mask=pore_mask, solid_mask=solid_mask, sample_id=sample_id, origin_zyx=origin_zyx)
        if SAVE_INDIVIDUAL_MECH_XLSX:
            write_mechanics_xlsx(mech_row, mech_xlsx)
        statuses = [mech_row.get("Ex_status", ""), mech_row.get("Ey_status", ""), mech_row.get("Ez_status", "")]
        if all(s == "ok" for s in statuses):
            status = "ok"
        elif any(str(s).startswith("not_percolating") for s in statuses):
            status = "ok_with_nonpercolating_solid_direction"
        else:
            status = "ok_with_directional_solver_warning"
    except Exception as e:
        mech_row = {"sample_id": sample_id, "z_start": origin_zyx[0], "y_start": origin_zyx[1], "x_start": origin_zyx[2], "block_size": BLOCK_SIZE}
        status = f"failed: {e}"
    mech_row["mechanics_xlsx"] = str(mech_xlsx) if SAVE_INDIVIDUAL_MECH_XLSX else ""
    mech_row["mechanics_status"] = status
    return mech_row


def compute_mechanics_dataset_v2():
    pore_csv = SUMMARY_OUT / f"{DATASET_PREFIX}_pore_summary_v2.csv"
    if not pore_csv.exists():
        raise FileNotFoundError("请先运行 Step 2：孔隙分割。")
    summary_csv = SUMMARY_OUT / f"{DATASET_PREFIX}_mechanics_summary_v2.csv"
    summary_xlsx = SUMMARY_OUT / f"{DATASET_PREFIX}_mechanics_summary_v2.xlsx"
    if summary_csv.exists() and SKIP_EXISTING_FILES and (not FORCE_REBUILD_DATASET):
        print("Step 4 已存在，直接读取:", summary_csv)
        return pd.read_csv(summary_csv)
    df = pd.read_csv(pore_csv)
    print("=" * 90)
    print("Step 4: 力学代理参数、固体连通与骨架特征计算开始")
    print("样本数量:", len(df))
    print("是否并行:", PARALLEL_MECHANICS, "任务数:", N_JOBS_MECHANICS)
    print("=" * 90)
    rows = df.to_dict("records")
    if PARALLEL_MECHANICS and len(rows) > 1:
        with tqdm_joblib(tqdm(total=len(rows), desc="力学参数计算", unit="样本")):
            summary_rows = Parallel(n_jobs=N_JOBS_MECHANICS, backend="loky", batch_size=1, verbose=0)(
                delayed(process_mechanics_row)(r) for r in rows
            )
    else:
        summary_rows = [process_mechanics_row(r) for r in tqdm(rows, desc="力学参数计算", unit="样本")]
    out_df = pd.DataFrame(summary_rows)
    out_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
    out_df.to_excel(summary_xlsx, index=False)
    out_df.to_csv(MECH_OUT / f"{DATASET_PREFIX}_mechanics_summary_v2.csv", index=False, encoding="utf-8-sig")
    out_df.to_excel(MECH_OUT / f"{DATASET_PREFIX}_mechanics_summary_v2.xlsx", index=False)
    print("Step 4 完成:", summary_csv)
    return out_df


def merge_master_dataset_v2():
    master_csv = SUMMARY_OUT / f"{DATASET_PREFIX}_GNN_dataset_v2_master.csv"
    if master_csv.exists() and SKIP_EXISTING_FILES and (not FORCE_REBUILD_DATASET):
        print("Master 已存在，直接读取:", master_csv)
        return pd.read_csv(master_csv)
    pore_csv = SUMMARY_OUT / f"{DATASET_PREFIX}_pore_summary_v2.csv"
    perm_csv = SUMMARY_OUT / f"{DATASET_PREFIX}_permeation_summary_v2.csv"
    mech_csv = SUMMARY_OUT / f"{DATASET_PREFIX}_mechanics_summary_v2.csv"
    pore_df = pd.read_csv(pore_csv)
    perm_df = pd.read_csv(perm_csv)
    mech_df = pd.read_csv(mech_csv)
    master = pore_df.merge(
        perm_df,
        on=["sample_id", "z_start", "y_start", "x_start", "block_size"],
        how="left",
        suffixes=("", "_perm")
    ).merge(
        mech_df,
        on=["sample_id", "z_start", "y_start", "x_start", "block_size"],
        how="left",
        suffixes=("", "_mech")
    )
    master_xlsx = SUMMARY_OUT / f"{DATASET_PREFIX}_GNN_dataset_v2_master.xlsx"
    master.to_csv(master_csv, index=False, encoding="utf-8-sig")
    master.to_excel(master_xlsx, index=False)
    print("最终 GNN 数据集总表已生成:", master_csv)
    print("样本数量:", len(master))
    return master


In [ ]:

# ============================================================
# 5. 一键生成 EF-78 页岩 的数据制作文件
# ============================================================
# 该单元会产生 raw/pore/Permeation/Mechanics/summary 文件夹。
# 如果已经完整生成过并且不想重算，可把 FORCE_REBUILD_DATASET=False 且 SKIP_EXISTING_FILES=True。

start_time = time.time()
blocks_df = crop_raw_blocks()
pore_summary_df = segment_pores_directional()
permeation_summary_df = compute_directional_permeability_dataset()
mechanics_summary_df = compute_mechanics_dataset_v2()
master_df = merge_master_dataset_v2()
print(f"数据制作总耗时: {(time.time() - start_time) / 60:.2f} min")

# 快速质量检查
check_cols = ["porosity", "connected_porosity", "kx_um2", "ky_um2", "kz_um2", "Ex_spring_GPa", "Ey_spring_GPa", "Ez_spring_GPa"]
check_cols = [c for c in check_cols if c in master_df.columns]
display(master_df[check_cols].describe())
print("贯通比例:")
for d in ["x", "y", "z"]:
    if f"percolates_{d}" in master_df.columns:
        print(f"percolates_{d}:", pd.to_numeric(master_df[f"percolates_{d}"], errors="coerce").mean())



## 6. 加载 PM-GNN 模型并构建预测输入

下面开始只读取 `Final-PM-GNN` 中的模型结果与 scaler，不训练模型。


In [ ]:

# ============================================================
# 6. PM-GNN 推理用特征、图构建函数和模型定义
# ============================================================

CONFIG_PATH = MODEL_DIR / "pm_gnn_flow_model_config.json"
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"找不到 PM-GNN 配置文件: {CONFIG_PATH}")
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PM_CONFIG = json.load(f)

NODE_FEATURE_COLS = PM_CONFIG["node_feature_cols"]
EDGE_FEATURE_COLS = PM_CONFIG["edge_feature_cols"]
TARGET_COLS = PM_CONFIG["target_cols"]
PERM_TARGET_COLS = ["kx_um2", "ky_um2", "kz_um2"]
MECH_TARGET_COLS = ["Ex_spring_GPa", "Ey_spring_GPa", "Ez_spring_GPa"]
PERM_IDXS = [TARGET_COLS.index(c) for c in PERM_TARGET_COLS]
MECH_IDXS = [TARGET_COLS.index(c) for c in MECH_TARGET_COLS]

BASE_GLOBAL_FEATURE_COLS = PM_CONFIG["base_global_feature_cols"]
FLOW_GLOBAL_FEATURE_COLS = PM_CONFIG["flow_global_feature_cols"]
PERM_GLOBAL_FEATURE_COLS = PM_CONFIG["perm_global_feature_cols"]
MECH_GLOBAL_FEATURE_COLS = PM_CONFIG["mech_global_feature_cols"]

EPS_K = 1e-12
FLOW_BOUNDARY_FRACTION = 0.05
FLOW_LOG_FLOOR = -30.0

print("NODE_FEATURE_COLS:", NODE_FEATURE_COLS)
print("EDGE_FEATURE_COLS:", EDGE_FEATURE_COLS)
print("PERM global dim:", len(PERM_GLOBAL_FEATURE_COLS))
print("MECH global dim:", len(MECH_GLOBAL_FEATURE_COLS))
print("TARGET_COLS:", TARGET_COLS)


def export_figure_data(data, figure_path, suffix='_data.csv'):
    figure_path = Path(figure_path)
    data_dir = FIG_DATA_OUT
    data_dir.mkdir(parents=True, exist_ok=True)
    suffix = suffix if str(suffix).startswith('_') else '_' + str(suffix)
    data_path = data_dir / f"{figure_path.stem}{suffix}"
    if isinstance(data, pd.DataFrame):
        df = data.copy()
    elif isinstance(data, dict):
        df = pd.DataFrame(data)
    else:
        df = pd.DataFrame(data)
    df.to_csv(data_path, index=False, encoding='utf-8-sig')
    return data_path


def transform_perm_y(y_perm):
    y_perm = np.asarray(y_perm, dtype=np.float32).copy()
    return np.log10(np.maximum(y_perm, 0) + EPS_K)


def inverse_perm_y(y_perm_scaled, perm_y_scaler):
    y_log = perm_y_scaler.inverse_transform(y_perm_scaled)
    y_real = np.maximum(10 ** y_log - EPS_K, 0)
    return y_real


def inverse_mech_y(y_mech_scaled, mech_y_scaler):
    return mech_y_scaler.inverse_transform(y_mech_scaled)


def is_valid_path_value(value):
    if value is None:
        return False
    if isinstance(value, float) and np.isnan(value):
        return False
    text = str(value).strip()
    return text != "" and text.lower() != "nan"


def filename_from_any_path(path_value):
    return str(path_value).replace("\\", "/").split("/")[-1]


def safe_int_value(value, default=0):
    val = pd.to_numeric(value, errors="coerce")
    if pd.isna(val):
        return default
    return int(val)


def safe_float_from_row(row, col, default=0.0):
    if col not in row.index:
        return default
    val = pd.to_numeric(row.get(col), errors="coerce")
    if pd.isna(val):
        return default
    return float(val)


def select_primary_pnm(row):
    for d in ["x", "y", "z"]:
        pnm_col = f"pnm_{d}_xlsx"
        per_col = f"percolates_{d}"
        if pnm_col in row.index and safe_int_value(row.get(per_col, 0), default=0) == 1 and is_valid_path_value(row.get(pnm_col)):
            return str(row.get(pnm_col)), d
    return "", "none"


def _safe_array_stats(values):
    arr = pd.to_numeric(pd.Series(values), errors="coerce").replace([np.inf, -np.inf], np.nan).dropna().values
    if len(arr) == 0:
        return 0.0, 0.0, 0.0
    return float(np.mean(arr)), float(np.std(arr)), float(np.max(arr))


def _safe_range(values):
    arr = pd.to_numeric(pd.Series(values), errors="coerce").replace([np.inf, -np.inf], np.nan).dropna().values
    if len(arr) == 0:
        return 0.0
    return float(np.max(arr) - np.min(arr))


def _flow_safe_log10(value, floor=FLOW_LOG_FLOOR):
    value = float(value)
    if not np.isfinite(value) or value <= 0:
        return float(floor)
    return float(max(np.log10(value), floor))


def _flow_boundary_nodes(values, fraction=FLOW_BOUNDARY_FRACTION):
    values = np.asarray(values, dtype=float)
    n = len(values)
    if n == 0:
        return np.array([], dtype=np.int64), np.array([], dtype=np.int64)
    order = np.argsort(values)
    k = max(1, int(np.ceil(n * fraction)))
    k = min(k, max(1, n // 4))
    inlet = order[:k].astype(np.int64)
    outlet = order[-k:].astype(np.int64)
    if n > 1:
        outlet = np.asarray([v for v in outlet if v not in set(inlet.tolist())], dtype=np.int64)
        if len(outlet) == 0:
            outlet = np.asarray([order[-1]], dtype=np.int64)
    return inlet, outlet


def _flow_union_find_labels(num_nodes, src, dst):
    parent = np.arange(num_nodes, dtype=np.int64)
    rank = np.zeros(num_nodes, dtype=np.int8)
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a
    def union(a, b):
        ra = find(int(a)); rb = find(int(b))
        if ra == rb:
            return
        if rank[ra] < rank[rb]:
            parent[ra] = rb
        elif rank[ra] > rank[rb]:
            parent[rb] = ra
        else:
            parent[rb] = ra
            rank[ra] += 1
    for u, v in zip(src, dst):
        if u != v:
            union(u, v)
    return np.asarray([find(i) for i in range(num_nodes)], dtype=np.int64)


def _flow_multisource_shortest(adj, sources, targets, weight_pos):
    import heapq
    if len(sources) == 0 or len(targets) == 0:
        return np.inf
    target_set = set(int(v) for v in targets)
    dist = np.full(len(adj), np.inf, dtype=float)
    heap = []
    for s in sources:
        s = int(s); dist[s] = 0.0; heapq.heappush(heap, (0.0, s))
    while heap:
        d_u, u = heapq.heappop(heap)
        if d_u != dist[u]:
            continue
        if u in target_set:
            return float(d_u)
        for item in adj[u]:
            v = item[0]; w = float(item[weight_pos])
            if not np.isfinite(w) or w < 0:
                continue
            nd = d_u + w
            if nd < dist[v]:
                dist[v] = nd
                heapq.heappush(heap, (nd, v))
    return np.inf


def _flow_multisource_widest(adj, sources, targets):
    import heapq
    if len(sources) == 0 or len(targets) == 0:
        return 0.0
    target_set = set(int(v) for v in targets)
    score = np.zeros(len(adj), dtype=float)
    heap = []
    for s in sources:
        s = int(s); score[s] = np.inf; heapq.heappush(heap, (-score[s], s))
    while heap:
        neg_score, u = heapq.heappop(heap)
        current = -neg_score
        if current < score[u]:
            continue
        if u in target_set:
            return float(current)
        for item in adj[u]:
            v = item[0]; conductance = float(item[3])
            candidate = min(current, conductance)
            if candidate > score[v]:
                score[v] = candidate
                heapq.heappush(heap, (-candidate, v))
    return 0.0


def compute_flow_path_descriptors(pores, throats):
    result = {name: 0.0 for name in FLOW_GLOBAL_FEATURE_COLS}
    required_pore_cols = ["X Coord [cm]", "Y Coord [cm]", "Z Coord [cm]"]
    required_throat_cols = ["Pore ID #1", "Pore ID #2", "EqRadius [cm]", "ChannelLength [cm]"]
    if any(c not in pores.columns for c in required_pore_cols) or any(c not in throats.columns for c in required_throat_cols):
        return result
    coords = pores[required_pore_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=float)
    n_nodes = len(coords)
    if n_nodes < 2:
        return result
    src = pd.to_numeric(throats["Pore ID #1"], errors="coerce").fillna(-1).to_numpy(dtype=np.int64)
    dst = pd.to_numeric(throats["Pore ID #2"], errors="coerce").fillna(-1).to_numpy(dtype=np.int64)
    radius = pd.to_numeric(throats["EqRadius [cm]"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
    channel_length = pd.to_numeric(throats["ChannelLength [cm]"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
    valid = (src >= 0) & (src < n_nodes) & (dst >= 0) & (dst < n_nodes) & (src != dst)
    src = src[valid]; dst = dst[valid]; radius = radius[valid]; channel_length = channel_length[valid]
    if len(src) == 0:
        return result
    delta = coords[dst] - coords[src]
    geometric_length = np.linalg.norm(delta, axis=1)
    eps = 1e-18
    effective_length = np.where(channel_length > eps, channel_length, geometric_length)
    effective_length = np.maximum(effective_length, eps)
    radius = np.maximum(radius, 0.0)
    conductance = np.power(radius, 4) / effective_length
    conductance = np.nan_to_num(conductance, nan=0.0, posinf=0.0, neginf=0.0)
    resistance = 1.0 / np.maximum(conductance, eps)
    positive_g = conductance[conductance > 0]
    if len(positive_g) > 0:
        q10, q50, q90 = np.quantile(positive_g, [0.10, 0.50, 0.90])
        result["flow_log10_conductance_p10"] = _flow_safe_log10(q10)
        result["flow_log10_conductance_p50"] = _flow_safe_log10(q50)
        result["flow_log10_conductance_p90"] = _flow_safe_log10(q90)
    else:
        result["flow_log10_conductance_p10"] = FLOW_LOG_FLOOR
        result["flow_log10_conductance_p50"] = FLOW_LOG_FLOOR
        result["flow_log10_conductance_p90"] = FLOW_LOG_FLOOR
    degree = np.zeros(n_nodes, dtype=np.int64)
    np.add.at(degree, src, 1); np.add.at(degree, dst, 1)
    result["flow_dead_end_fraction"] = float(np.mean(degree <= 1))
    labels = _flow_union_find_labels(n_nodes, src, dst)
    adj = [[] for _ in range(n_nodes)]
    for u, v, gl, r_e, g_e in zip(src, dst, geometric_length, resistance, conductance):
        u = int(u); v = int(v)
        adj[u].append((v, float(max(gl, eps)), float(max(r_e, eps)), float(max(g_e, 0.0))))
        adj[v].append((u, float(max(gl, eps)), float(max(r_e, eps)), float(max(g_e, 0.0))))
    safe_geom = np.maximum(geometric_length, eps)
    direction_cos_sq = np.square(delta / safe_geom[:, None])
    for axis, axis_name in enumerate(["x", "y", "z"]):
        span = float(np.max(coords[:, axis]) - np.min(coords[:, axis]))
        if span <= eps:
            result[f"flow_{axis_name}_path_exists"] = 0.0
            result[f"flow_{axis_name}_backbone_node_fraction"] = 0.0
            result[f"flow_{axis_name}_tortuosity"] = 0.0
            result[f"flow_{axis_name}_log10_best_path_conductance"] = FLOW_LOG_FLOOR
            result[f"flow_{axis_name}_log10_widest_bottleneck"] = FLOW_LOG_FLOOR
            result[f"flow_{axis_name}_log10_directional_conductance_sum"] = FLOW_LOG_FLOOR
            continue
        inlet, outlet = _flow_boundary_nodes(coords[:, axis])
        percolating_labels = set(labels[inlet].tolist()).intersection(set(labels[outlet].tolist()))
        path_exists = len(percolating_labels) > 0
        result[f"flow_{axis_name}_path_exists"] = float(path_exists)
        if path_exists:
            result[f"flow_{axis_name}_backbone_node_fraction"] = float(np.mean(np.isin(labels, list(percolating_labels))))
        else:
            result[f"flow_{axis_name}_backbone_node_fraction"] = 0.0
        shortest_geom = _flow_multisource_shortest(adj, inlet, outlet, weight_pos=1)
        result[f"flow_{axis_name}_tortuosity"] = float(max(shortest_geom / span, 1.0)) if np.isfinite(shortest_geom) and span > eps else 0.0
        shortest_resistance = _flow_multisource_shortest(adj, inlet, outlet, weight_pos=2)
        best_path_conductance = 1.0 / shortest_resistance if np.isfinite(shortest_resistance) and shortest_resistance > 0 else 0.0
        result[f"flow_{axis_name}_log10_best_path_conductance"] = _flow_safe_log10(best_path_conductance)
        result[f"flow_{axis_name}_log10_widest_bottleneck"] = _flow_safe_log10(_flow_multisource_widest(adj, inlet, outlet))
        directional_sum = float(np.sum(conductance * direction_cos_sq[:, axis]))
        result[f"flow_{axis_name}_log10_directional_conductance_sum"] = _flow_safe_log10(directional_sum)
    return result


def read_pnm_graph(row):
    pnm_path_value, primary_dir = select_primary_pnm(row)
    if not pnm_path_value:
        return None
    pnm_path = Path(pnm_path_value)
    if not pnm_path.exists():
        pnm_path = PORE_OUT / filename_from_any_path(pnm_path_value)
    if not pnm_path.exists():
        return None
    try:
        pores = pd.read_excel(pnm_path, sheet_name="Pores")
        throats = pd.read_excel(pnm_path, sheet_name="Throats")
    except Exception:
        return None
    if len(pores) == 0 or len(throats) == 0:
        return None
    if any(c not in pores.columns for c in NODE_FEATURE_COLS) or any(c not in throats.columns for c in EDGE_FEATURE_COLS):
        return None
    x = pores[NODE_FEATURE_COLS].replace([np.inf, -np.inf], np.nan).fillna(0).values.astype(np.float32)
    src = pd.to_numeric(throats["Pore ID #1"], errors="coerce").fillna(-1).values.astype(np.int64)
    dst = pd.to_numeric(throats["Pore ID #2"], errors="coerce").fillna(-1).values.astype(np.int64)
    edge_index = np.vstack([np.concatenate([src, dst]), np.concatenate([dst, src])]).T
    e = throats[EDGE_FEATURE_COLS].replace([np.inf, -np.inf], np.nan).fillna(0).values.astype(np.float32)
    edge_attr = np.concatenate([e, e], axis=0)
    n = x.shape[0]
    valid = (edge_index[:, 0] >= 0) & (edge_index[:, 0] < n) & (edge_index[:, 1] >= 0) & (edge_index[:, 1] < n)
    edge_index = edge_index[valid]
    edge_attr = edge_attr[valid]
    if len(edge_index) == 0:
        return None

    target_values = []
    has_true = True
    for col in TARGET_COLS:
        val = pd.to_numeric(row.get(col, np.nan), errors="coerce")
        if pd.isna(val) or not np.isfinite(val):
            has_true = False
            val = np.nan
        target_values.append(float(val))

    num_pores_calc = int(x.shape[0])
    num_throats_calc = int(len(edge_index) // 2)
    pore_radius = pd.to_numeric(pores["EqRadius [cm]"], errors="coerce") if "EqRadius [cm]" in pores.columns else pd.Series(dtype=float)
    throat_radius = pd.to_numeric(throats["EqRadius [cm]"], errors="coerce") if "EqRadius [cm]" in throats.columns else pd.Series(dtype=float)
    throat_length = pd.to_numeric(throats["ChannelLength [cm]"], errors="coerce") if "ChannelLength [cm]" in throats.columns else pd.Series(dtype=float)
    degree = np.bincount(edge_index[:, 0], minlength=num_pores_calc).astype(float)
    mean_degree, std_degree, max_degree = _safe_array_stats(degree)
    mean_pore_r, std_pore_r, max_pore_r = _safe_array_stats(pore_radius)
    mean_throat_r, std_throat_r, max_throat_r = _safe_array_stats(throat_radius)
    mean_throat_l, std_throat_l, max_throat_l = _safe_array_stats(throat_length)
    bbox_x = _safe_range(pores["X Coord [cm]"])
    bbox_y = _safe_range(pores["Y Coord [cm]"])
    bbox_z = _safe_range(pores["Z Coord [cm]"])
    bbox_volume = float(max(bbox_x, 1e-12) * max(bbox_y, 1e-12) * max(bbox_z, 1e-12))
    flow_global = compute_flow_path_descriptors(pores, throats)
    computed = {
        "graph_num_pores": float(num_pores_calc),
        "graph_num_throats": float(num_throats_calc),
        "graph_edge_density": float(num_throats_calc / max(num_pores_calc * (num_pores_calc - 1) / 2, 1.0)),
        "graph_mean_degree": mean_degree,
        "graph_std_degree": std_degree,
        "graph_max_degree": max_degree,
        "graph_mean_pore_radius": mean_pore_r,
        "graph_std_pore_radius": std_pore_r,
        "graph_max_pore_radius": max_pore_r,
        "graph_mean_throat_radius": mean_throat_r,
        "graph_std_throat_radius": std_throat_r,
        "graph_max_throat_radius": max_throat_r,
        "graph_mean_throat_length": mean_throat_l,
        "graph_std_throat_length": std_throat_l,
        "graph_max_throat_length": max_throat_l,
        "graph_bbox_x": bbox_x,
        "graph_bbox_y": bbox_y,
        "graph_bbox_z": bbox_z,
        "graph_bbox_volume": bbox_volume,
        "primary_dir_is_x": 1.0 if primary_dir == "x" else 0.0,
        "primary_dir_is_y": 1.0 if primary_dir == "y" else 0.0,
        "primary_dir_is_z": 1.0 if primary_dir == "z" else 0.0,
    }
    computed.update(flow_global)

    feature_dict = {}
    for col in set(PERM_GLOBAL_FEATURE_COLS + MECH_GLOBAL_FEATURE_COLS):
        if col in computed:
            val = computed[col]
        else:
            val = safe_float_from_row(row, col, default=0.0)
        if not np.isfinite(val):
            val = 0.0
        feature_dict[col] = float(val)

    return {
        "sample_id": row.get("sample_id", "unknown"),
        "rock_id": row.get("rock_id", DATASET_PREFIX),
        "x": x,
        "edge_index": edge_index.astype(np.int64),
        "edge_attr": edge_attr.astype(np.float32),
        "global_feature_dict": feature_dict,
        "y": np.asarray(target_values, dtype=np.float32),
        "has_true": bool(has_true),
        "pnm_xlsx": str(pnm_path),
        "primary_pnm_direction": primary_dir,
    }


In [ ]:

# ============================================================
# 7. 加载 scaler，构建图数据并标准化
# ============================================================

def load_sklearn_artifact(path):
    """
    兼容读取 sklearn/joblib/pickle 保存的 scaler。
    之前第 7 步直接用 pickle.load，部分环境下 joblib 压缩文件会报：
    UnpicklingError: invalid load key, '\x08'
    因此这里优先使用 joblib.load，并保留 pickle 作为兜底。
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"找不到模型标准化文件: {path}")
    errors = []
    try:
        return joblib.load(path)
    except Exception as e:
        errors.append(f"joblib.load: {type(e).__name__}: {e}")
    try:
        with open(path, "rb") as f:
            return pickle.load(f)
    except Exception as e:
        errors.append(f"pickle.load: {type(e).__name__}: {e}")

    head = path.read_bytes()[:16]
    head_hex = " ".join(f"{b:02x}" for b in head)
    raise RuntimeError(
        "scaler 文件无法读取，通常是 Final-PM-GNN 没有完整解压、路径指向错误、"
        "文件被网盘/压缩软件损坏，或正在读取压缩包内部的占位文件。\n"
        f"文件: {path}\n"
        f"文件大小: {path.stat().st_size} bytes\n"
        f"文件头16字节: {head_hex}\n"
        "建议：删除当前 Final-PM-GNN 文件夹，重新把 Final-PM-GNN(1).zip 完整解压到 "
        "checkpoints/PM-GNN_saved_tag_9train_2val 后再运行。\n"
        + "\n".join(errors)
    )

node_scaler = load_sklearn_artifact(MODEL_DIR / "node_scaler.pkl")
edge_scaler = load_sklearn_artifact(MODEL_DIR / "edge_scaler.pkl")
perm_global_feature_scaler = load_sklearn_artifact(MODEL_DIR / "perm_global_feature_scaler.pkl")
mech_global_feature_scaler = load_sklearn_artifact(MODEL_DIR / "mech_global_feature_scaler.pkl")
perm_y_scaler = load_sklearn_artifact(MODEL_DIR / "perm_y_scaler.pkl")
mech_y_scaler = load_sklearn_artifact(MODEL_DIR / "mech_y_scaler.pkl")

print("已成功加载全部 scaler。")

master_path = SUMMARY_OUT / f"{DATASET_PREFIX}_GNN_dataset_v2_master.csv"
if not master_path.exists():
    raise FileNotFoundError(f"找不到 master 数据表: {master_path}。请先运行数据制作单元。")
master_df = pd.read_csv(master_path)

# 清洗：需要有可用 PNM；真实指标需要 6 个标签有限。
for col in TARGET_COLS:
    if col in master_df.columns:
        master_df[col] = pd.to_numeric(master_df[col], errors="coerce")

master_df = master_df.replace([np.inf, -np.inf], np.nan)
print("master 样本数:", len(master_df))

raw_graphs = []
failed = []
for _, row in tqdm(master_df.iterrows(), total=len(master_df), desc="读取 PNM 并构建 PM-GNN 图"):
    g = read_pnm_graph(row)
    if g is None:
        failed.append(row.get("sample_id", "unknown"))
    else:
        raw_graphs.append(g)

print("可预测图数量:", len(raw_graphs))
print("跳过图数量:", len(failed))
if failed[:10]:
    print("前10个跳过样本:", failed[:10])
if len(raw_graphs) == 0:
    raise ValueError("没有可用于 PM-GNN 预测的 PNM 图。请检查贯通孔隙与 PNM 文件是否生成。")


def apply_saved_scalers_to_graph(g):
    node_scaled = node_scaler.transform(g["x"]).astype(np.float32)
    edge_scaled = edge_scaler.transform(g["edge_attr"]).astype(np.float32)
    perm_global = np.asarray([g["global_feature_dict"].get(c, 0.0) for c in PERM_GLOBAL_FEATURE_COLS], dtype=np.float32).reshape(1, -1)
    mech_global = np.asarray([g["global_feature_dict"].get(c, 0.0) for c in MECH_GLOBAL_FEATURE_COLS], dtype=np.float32).reshape(1, -1)
    perm_global = np.nan_to_num(perm_global, nan=0.0, posinf=0.0, neginf=0.0)
    mech_global = np.nan_to_num(mech_global, nan=0.0, posinf=0.0, neginf=0.0)
    perm_global_scaled = perm_global_feature_scaler.transform(perm_global).astype(np.float32).reshape(-1)
    mech_global_scaled = mech_global_feature_scaler.transform(mech_global).astype(np.float32).reshape(-1)
    g["x_perm_scaled"] = np.concatenate([node_scaled, np.repeat(perm_global_scaled.reshape(1, -1), node_scaled.shape[0], axis=0)], axis=1).astype(np.float32)
    g["x_mech_scaled"] = np.concatenate([node_scaled, np.repeat(mech_global_scaled.reshape(1, -1), node_scaled.shape[0], axis=0)], axis=1).astype(np.float32)
    g["edge_attr_scaled"] = edge_scaled
    if g.get("has_true", False):
        y_all = g["y"].reshape(1, -1)
        g["y_perm_scaled"] = perm_y_scaler.transform(transform_perm_y(y_all[:, PERM_IDXS])).astype(np.float32).reshape(-1)
        g["y_mech_scaled"] = mech_y_scaler.transform(y_all[:, MECH_IDXS]).astype(np.float32).reshape(-1)
    else:
        g["y_perm_scaled"] = np.full(len(PERM_TARGET_COLS), np.nan, dtype=np.float32)
        g["y_mech_scaled"] = np.full(len(MECH_TARGET_COLS), np.nan, dtype=np.float32)
    return g


graphs = [apply_saved_scalers_to_graph(g) for g in raw_graphs]

feature_rows = []
for g in graphs:
    row = {"sample_id": g["sample_id"], "primary_pnm_direction": g["primary_pnm_direction"], "pnm_xlsx": g["pnm_xlsx"]}
    row.update(g["global_feature_dict"])
    feature_rows.append(row)
feature_df = pd.DataFrame(feature_rows)
feature_df.to_csv(PRED_OUT / "pm_gnn_input_global_features.csv", index=False, encoding="utf-8-sig")

print("渗流模型节点输入维度:", graphs[0]["x_perm_scaled"].shape[1])
print("力学模型节点输入维度:", graphs[0]["x_mech_scaled"].shape[1])
print("全局特征表已保存:", PRED_OUT / "pm_gnn_input_global_features.csv")


In [ ]:

# ============================================================
# 8. 定义并加载 PM-GNN 权重
# ============================================================

def add_self_loops(edge_index, num_nodes):
    self_edges = tf.stack([tf.range(num_nodes), tf.range(num_nodes)], axis=1)
    self_edges = tf.cast(self_edges, tf.int32)
    return tf.concat([edge_index, self_edges], axis=0)


def normalize_messages(messages, dst, num_nodes):
    out = tf.math.unsorted_segment_sum(messages, dst, num_nodes)
    deg = tf.math.unsorted_segment_sum(tf.ones((tf.shape(dst)[0], 1), dtype=tf.float32), dst, num_nodes)
    return out / tf.maximum(deg, 1.0)


class BasicGraphConv(layers.Layer):
    def __init__(self, hidden_dim, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.dense = layers.Dense(hidden_dim, activation="relu")
        self.dropout = layers.Dropout(dropout)

    def call(self, h, edge_index, edge_attr=None, training=False):
        num_nodes = tf.shape(h)[0]
        edge_index2 = add_self_loops(edge_index, num_nodes)
        src = edge_index2[:, 0]
        dst = edge_index2[:, 1]
        h_src = tf.gather(h, src)
        agg = normalize_messages(h_src, dst, num_nodes)
        h_new = self.dense(agg, training=training)
        h_new = self.dropout(h_new, training=training)
        return h_new


class PMGNN(keras.Model):
    def __init__(self, node_dim, hidden_dim=64, num_layers=2, out_dim=3, dropout=0.1, direction_conditioned=False):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.out_dim = out_dim
        self.direction_conditioned = direction_conditioned
        self.encoder = keras.Sequential([layers.Dense(hidden_dim, activation="relu"), layers.Dropout(dropout)])
        self.convs = [BasicGraphConv(hidden_dim, dropout=dropout) for _ in range(num_layers)]
        if self.direction_conditioned:
            if out_dim != 3:
                raise ValueError("方向条件化渗流头要求 out_dim=3。")
            self.direction_encoder = keras.Sequential([layers.Dense(16, activation="relu")])
            self.scalar_readout = keras.Sequential([layers.Dense(hidden_dim, activation="relu"), layers.Dropout(dropout), layers.Dense(1)])
            self.readout = None
        else:
            self.direction_encoder = None
            self.scalar_readout = None
            self.readout = keras.Sequential([layers.Dense(hidden_dim, activation="relu"), layers.Dropout(dropout), layers.Dense(out_dim)])

    def call(self, x, edge_index, edge_attr=None, batch_index=None, training=False, return_embedding=False):
        h = self.encoder(x, training=training)
        for conv in self.convs:
            h = conv(h, edge_index, edge_attr=edge_attr, training=training)
        if batch_index is not None:
            graph_emb = tf.math.segment_mean(h, batch_index)
        else:
            graph_emb = tf.reduce_mean(h, axis=0, keepdims=True)
        if self.direction_conditioned:
            batch_size = tf.shape(graph_emb)[0]
            direction_one_hot = tf.eye(3, dtype=graph_emb.dtype)
            direction_one_hot = tf.tile(direction_one_hot, [batch_size, 1])
            graph_repeated = tf.repeat(graph_emb, repeats=3, axis=0)
            direction_emb = self.direction_encoder(direction_one_hot, training=training)
            fused = tf.concat([graph_repeated, direction_emb], axis=1)
            scalar_pred = self.scalar_readout(fused, training=training)
            y_pred = tf.reshape(scalar_pred, [batch_size, 3])
        else:
            y_pred = self.readout(graph_emb, training=training)
        if return_embedding:
            return y_pred, graph_emb
        return y_pred


def graph_to_tensors(g, task="perm"):
    if task == "perm":
        x_np = g["x_perm_scaled"]
        y_np = g["y_perm_scaled"]
    elif task == "mech":
        x_np = g["x_mech_scaled"]
        y_np = g["y_mech_scaled"]
    else:
        raise ValueError("task 只能是 'perm' 或 'mech'")
    x = tf.convert_to_tensor(x_np, dtype=tf.float32)
    edge_index = tf.convert_to_tensor(g["edge_index"], dtype=tf.int32)
    edge_attr = tf.convert_to_tensor(g["edge_attr_scaled"], dtype=tf.float32)
    y = tf.convert_to_tensor(y_np, dtype=tf.float32)
    return x, edge_index, edge_attr, y


def batch_graphs_for_prediction(graph_list, task):
    x_list, edge_index_list, edge_attr_list, y_list, batch_index_list = [], [], [], [], []
    node_offset = 0
    for i, g in enumerate(graph_list):
        x, edge_index, edge_attr, y = graph_to_tensors(g, task=task)
        edge_index = edge_index + node_offset
        x_list.append(x)
        edge_index_list.append(edge_index)
        edge_attr_list.append(edge_attr)
        y_list.append(y)
        batch_index_list.append(tf.fill((tf.shape(x)[0],), i))
        node_offset += int(tf.shape(x)[0])
    return tf.concat(x_list, axis=0), tf.concat(edge_index_list, axis=0), tf.concat(edge_attr_list, axis=0), tf.stack(y_list, axis=0), tf.concat(batch_index_list, axis=0)


hidden_dim = int(PM_CONFIG.get("hidden_dim", 64))
num_layers = int(PM_CONFIG.get("num_layers", 2))
dropout = float(PM_CONFIG.get("dropout", 0.1))
direction_conditioned = bool(PM_CONFIG.get("use_direction_conditioned_perm_head", False))

perm_model = PMGNN(
    node_dim=len(NODE_FEATURE_COLS) + len(PERM_GLOBAL_FEATURE_COLS),
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    out_dim=len(PERM_TARGET_COLS),
    dropout=dropout,
    direction_conditioned=direction_conditioned,
)
mech_model = PMGNN(
    node_dim=len(NODE_FEATURE_COLS) + len(MECH_GLOBAL_FEATURE_COLS),
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    out_dim=len(MECH_TARGET_COLS),
    dropout=dropout,
    direction_conditioned=False,
)

# 用第一个图 build 模型，再加载 .weights.h5。
x0, ei0, ea0, y0 = graph_to_tensors(graphs[0], task="perm")
_ = perm_model(x0, ei0, ea0, training=False)
x0, ei0, ea0, y0 = graph_to_tensors(graphs[0], task="mech")
_ = mech_model(x0, ei0, ea0, training=False)

perm_weights = MODEL_DIR / "pore_topology_pm_gnn_flow_path_descriptors_perm_rock_group_9train_2val.weights.h5"
mech_weights = MODEL_DIR / "pore_topology_pm_gnn_flow_path_descriptors_mech_rock_group_9train_2val.weights.h5"
if not perm_weights.exists() or not mech_weights.exists():
    # 兜底自动搜索，防止 split 名称不同。
    perm_candidates = sorted(MODEL_DIR.glob("*perm*.weights.h5"))
    mech_candidates = sorted(MODEL_DIR.glob("*mech*.weights.h5"))
    if not perm_candidates or not mech_candidates:
        raise FileNotFoundError("在 MODEL_DIR 中找不到 PM-GNN 的 perm/mech 权重文件。")
    perm_weights = perm_candidates[0]
    mech_weights = mech_candidates[0]

perm_model.load_weights(str(perm_weights))
mech_model.load_weights(str(mech_weights))
print("已加载渗流模型权重:", perm_weights)
print("已加载力学模型权重:", mech_weights)


In [ ]:

# ============================================================
# 9. 对 EF-78.raw 的所有可用子块进行 PM-GNN 预测
# ============================================================

def make_memory_safe_batches(graph_list, max_graphs=32, max_nodes=80000, max_edges=200000):
    batches, current = [], []
    n_nodes = 0
    n_edges = 0
    for g in graph_list:
        g_nodes = int(g["x"].shape[0])
        g_edges = int(g["edge_index"].shape[0])
        if current and (len(current) >= max_graphs or n_nodes + g_nodes > max_nodes or n_edges + g_edges > max_edges):
            batches.append(current)
            current, n_nodes, n_edges = [], 0, 0
        current.append(g)
        n_nodes += g_nodes
        n_edges += g_edges
    if current:
        batches.append(current)
    return batches


def predict_scaled_for_graphs(model, graph_list, task="perm", batch_size=32):
    preds_scaled, y_scaled, sample_ids, rock_ids = [], [], [], []
    batches = make_memory_safe_batches(graph_list, max_graphs=batch_size)
    for batch in tqdm(batches, desc=f"PM-GNN {task} 预测"):
        x, edge_index, edge_attr, y, batch_index = batch_graphs_for_prediction(batch, task=task)
        pred = model(x, edge_index, edge_attr, batch_index, training=False).numpy()
        preds_scaled.append(pred)
        y_scaled.append(y.numpy())
        for g in batch:
            sample_ids.append(g["sample_id"])
            rock_ids.append(g.get("rock_id", DATASET_PREFIX))
    return sample_ids, rock_ids, np.vstack(y_scaled), np.vstack(preds_scaled)


def predict_perm_graphs(graph_list):
    sample_ids, rock_ids, y_scaled, preds_scaled = predict_scaled_for_graphs(perm_model, graph_list, task="perm", batch_size=32)
    preds_real = inverse_perm_y(preds_scaled, perm_y_scaler)
    if np.all(np.isfinite(y_scaled)):
        y_real = inverse_perm_y(y_scaled, perm_y_scaler)
    else:
        y_real = np.vstack([g["y"][PERM_IDXS] for g in graph_list]).astype(float)
    return sample_ids, rock_ids, y_real, preds_real


def predict_mech_graphs(graph_list):
    sample_ids, rock_ids, y_scaled, preds_scaled = predict_scaled_for_graphs(mech_model, graph_list, task="mech", batch_size=32)
    preds_real = inverse_mech_y(preds_scaled, mech_y_scaler)
    if np.all(np.isfinite(y_scaled)):
        y_real = inverse_mech_y(y_scaled, mech_y_scaler)
    else:
        y_real = np.vstack([g["y"][MECH_IDXS] for g in graph_list]).astype(float)
    return sample_ids, rock_ids, y_real, preds_real

sample_ids_perm, rock_ids_perm, y_true_perm, y_pred_perm = predict_perm_graphs(graphs)
sample_ids_mech, rock_ids_mech, y_true_mech, y_pred_mech = predict_mech_graphs(graphs)

sample_ids = sample_ids_perm
y_true = np.hstack([y_true_perm, y_true_mech])
y_pred = np.hstack([y_pred_perm, y_pred_mech])

result_df = pd.DataFrame({
    "sample_id": sample_ids,
    "rock_id": rock_ids_perm,
    "data_source": str(RAW_PATH),
    "model_dir": str(MODEL_DIR),
    "primary_pnm_direction": [g.get("primary_pnm_direction", "") for g in graphs],
    "pnm_xlsx": [g.get("pnm_xlsx", "") for g in graphs],
})
for i, col in enumerate(PERM_TARGET_COLS):
    result_df[f"{col}_真实值"] = y_true_perm[:, i]
    result_df[f"{col}_预测值"] = y_pred_perm[:, i]
    result_df[f"{col}_残差"] = y_pred_perm[:, i] - y_true_perm[:, i]
for i, col in enumerate(MECH_TARGET_COLS):
    result_df[f"{col}_真实值"] = y_true_mech[:, i]
    result_df[f"{col}_预测值"] = y_pred_mech[:, i]
    result_df[f"{col}_残差"] = y_pred_mech[:, i] - y_true_mech[:, i]

# 同时生成第 12 步需要的英文列名版本，避免 results_df 未定义。
# pred_* 是 PM-GNN 预测值；true_* 是本 notebook 中用于作图/指标的代理真实值。
results_df = result_df[["sample_id", "rock_id", "data_source", "model_dir", "primary_pnm_direction", "pnm_xlsx"]].copy()
for i, col in enumerate(TARGET_COLS):
    results_df[f"true_{col}"] = y_true[:, i]
    results_df[f"pred_{col}"] = y_pred[:, i]
    results_df[f"residual_{col}"] = y_pred[:, i] - y_true[:, i]

prediction_csv = PRED_OUT / f"{DATASET_PREFIX}_PM_GNN_prediction_results.csv"
prediction_xlsx = PRED_OUT / f"{DATASET_PREFIX}_PM_GNN_prediction_results.xlsx"
prediction_csv_english = PRED_OUT / f"{DATASET_PREFIX}_PM_GNN_prediction_results_english_columns.csv"
prediction_xlsx_english = PRED_OUT / f"{DATASET_PREFIX}_PM_GNN_prediction_results_english_columns.xlsx"

result_df.to_csv(prediction_csv, index=False, encoding="utf-8-sig")
result_df.to_excel(prediction_xlsx, index=False)
results_df.to_csv(prediction_csv_english, index=False, encoding="utf-8-sig")
results_df.to_excel(prediction_xlsx_english, index=False)

print("预测结果已保存:", prediction_csv)
print("英文列名预测结果已保存:", prediction_csv_english)
display(result_df.head())
display(results_df.head())


In [ ]:

# ============================================================
# 10. 输出 PM-GNN 评价指标
# ============================================================

def safe_pearsonr(y_true_i, y_pred_i):
    if len(y_true_i) <= 2:
        return np.nan, np.nan
    if np.nanstd(y_true_i) == 0 or np.nanstd(y_pred_i) == 0:
        return np.nan, np.nan
    try:
        return pearsonr(y_true_i, y_pred_i)
    except Exception:
        return np.nan, np.nan


def safe_round(x, ndigits):
    try:
        if pd.isna(x):
            return np.nan
        return round(float(x), ndigits)
    except Exception:
        return np.nan


def calculate_regression_metrics(y_true_i, y_pred_i, col_name, n_features=10):
    y_true_i = np.asarray(y_true_i, dtype=float)
    y_pred_i = np.asarray(y_pred_i, dtype=float)
    valid = np.isfinite(y_true_i) & np.isfinite(y_pred_i)
    y_true_i = y_true_i[valid]
    y_pred_i = y_pred_i[valid]
    n_samples = len(y_true_i)
    if n_samples == 0:
        return {"预测目标": col_name, "样本数": 0}
    r2 = r2_score(y_true_i, y_pred_i) if n_samples > 1 else np.nan
    mae = mean_absolute_error(y_true_i, y_pred_i)
    rmse = np.sqrt(mean_squared_error(y_true_i, y_pred_i))
    r, p_value = safe_pearsonr(y_true_i, y_pred_i)
    explained_var = explained_variance_score(y_true_i, y_pred_i) if n_samples > 1 else np.nan
    medae = median_absolute_error(y_true_i, y_pred_i)
    maxae = max_error(y_true_i, y_pred_i)
    if n_samples > n_features + 1 and np.isfinite(r2):
        adj_r2 = 1 - (1 - r2) * (n_samples - 1) / (n_samples - n_features - 1)
    else:
        adj_r2 = np.nan
    epsilon = 1e-8
    mape = np.mean(np.abs((y_true_i - y_pred_i) / (y_true_i + epsilon))) * 100
    smape = np.mean(2 * np.abs(y_true_i - y_pred_i) / (np.abs(y_true_i) + np.abs(y_pred_i) + epsilon)) * 100
    rrmse = (rmse / (np.mean(y_true_i) + epsilon)) * 100
    bias = np.mean(y_pred_i) - np.mean(y_true_i)
    return {
        "预测目标": col_name,
        "样本数": n_samples,
        "R²": safe_round(r2, 4),
        "调整R²": safe_round(adj_r2, 4),
        "皮尔逊r": safe_round(r, 4),
        "p值": p_value,
        "解释方差": safe_round(explained_var, 4),
        "MAE": safe_round(mae, 6),
        "RMSE": safe_round(rmse, 6),
        "MedAE": safe_round(medae, 6),
        "MaxAE": safe_round(maxae, 6),
        "MAPE(%)": safe_round(mape, 2),
        "SMAPE(%)": safe_round(smape, 2),
        "RRMSE(%)": safe_round(rrmse, 2),
        "预测偏差": safe_round(bias, 6),
    }

metrics = []
for i, col in enumerate(TARGET_COLS):
    metrics.append(calculate_regression_metrics(y_true[:, i], y_pred[:, i], col, n_features=10))

numeric_metric_cols = ["R²", "调整R²", "皮尔逊r", "解释方差", "MAE", "RMSE", "MedAE", "MaxAE", "MAPE(%)", "SMAPE(%)", "RRMSE(%)", "预测偏差"]
avg_metrics = {"预测目标": "平均", "样本数": np.mean([m.get("样本数", np.nan) for m in metrics]), "p值": "-"}
for c in numeric_metric_cols:
    avg_metrics[c] = safe_round(np.nanmean([m.get(c, np.nan) for m in metrics]), 6 if c in ["MAE", "RMSE", "MedAE", "MaxAE", "预测偏差"] else 4)
metrics.append(avg_metrics)
metrics_df = pd.DataFrame(metrics)
metrics_df["p值"] = metrics_df["p值"].apply(lambda x: f"{x:.2e}" if isinstance(x, (float, np.floating)) and np.isfinite(x) else x)

metrics_csv = PRED_OUT / "metrics_gnn_complete_academic_pm_gnn_EF-78_model.csv"
metrics_xlsx = PRED_OUT / "metrics_gnn_complete_academic_pm_gnn_EF-78_model.xlsx"
metrics_df.to_csv(metrics_csv, index=False, encoding="utf-8-sig")
metrics_df.to_excel(metrics_xlsx, index=False)
print("评价指标已保存:", metrics_csv)
display(metrics_df)


In [ ]:

# ============================================================
# 11. 输出六张 true vs. predicted 图及其数据
# ============================================================
TARGET_DISPLAY_NAMES = {
    "kx_um2": "X-direction permeability",
    "ky_um2": "Y-direction permeability",
    "kz_um2": "Z-direction permeability",
    "Ex_spring_GPa": "X-direction equivalent elastic modulus",
    "Ey_spring_GPa": "Y-direction equivalent elastic modulus",
    "Ez_spring_GPa": "Z-direction equivalent elastic modulus",
}

figure_paths = []
for i, col in enumerate(TARGET_COLS):
    true_i = y_true[:, i].astype(float)
    pred_i = y_pred[:, i].astype(float)
    valid = np.isfinite(true_i) & np.isfinite(pred_i)
    true_i = true_i[valid]
    pred_i = pred_i[valid]

    fig_path = PRED_OUT / f"true_vs_predicted_{col}.png"
    fig_data = pd.DataFrame({
        "sample_id": np.asarray(sample_ids)[valid],
        "target": col,
        "true_value": true_i,
        "predicted_value": pred_i,
        "residual": pred_i - true_i,
    })
    data_path = export_figure_data(fig_data, fig_path)

    plt.figure(figsize=(5.5, 5))
    plt.scatter(true_i, pred_i, alpha=0.75)
    min_v = min(true_i.min(), pred_i.min())
    max_v = max(true_i.max(), pred_i.max())
    plt.plot([min_v, max_v], [min_v, max_v], linestyle="--")
    plt.xlabel("True value")
    plt.ylabel("Predicted value")
    plt.title(f"{TARGET_DISPLAY_NAMES[col]}: true vs. predicted")
    plt.tight_layout()
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()

    figure_paths.append({"target": col, "figure": str(fig_path), "data": str(data_path)})

figure_index_df = pd.DataFrame(figure_paths)
figure_index_df.to_csv(PRED_OUT / "true_vs_predicted_figure_index.csv", index=False, encoding="utf-8-sig")
print("六张图及其数据已保存到:", PRED_OUT)
display(figure_index_df)


In [ ]:
# ============================================================
# 12. 输出整块 EF-78 的代表性预测结果汇总
# ============================================================

# 兼容旧版本：如果第 9 步只生成了 result_df，这里自动重建 results_df。
if "results_df" not in globals():
    if "result_df" not in globals():
        raise NameError("未找到 results_df 或 result_df。请先完整运行第 9 步预测单元。")
    results_df = result_df[["sample_id", "rock_id", "data_source", "model_dir", "primary_pnm_direction", "pnm_xlsx"]].copy()
    for i, col in enumerate(TARGET_COLS):
        if f"{col}_预测值" in result_df.columns:
            results_df[f"pred_{col}"] = result_df[f"{col}_预测值"]
        elif "y_pred" in globals():
            results_df[f"pred_{col}"] = y_pred[:, i]
        else:
            results_df[f"pred_{col}"] = np.nan

        if f"{col}_真实值" in result_df.columns:
            results_df[f"true_{col}"] = result_df[f"{col}_真实值"]
        elif "y_true" in globals():
            results_df[f"true_{col}"] = y_true[:, i]
        else:
            results_df[f"true_{col}"] = np.nan

        results_df[f"residual_{col}"] = results_df[f"pred_{col}"] - results_df[f"true_{col}"]

summary_rows = []
for col in TARGET_COLS:
    pred_col = f"pred_{col}"
    true_col = f"true_{col}"
    if pred_col not in results_df.columns:
        print(f"警告：缺少 {pred_col}，跳过 {col}")
        continue

    vals = results_df[pred_col].astype(float).replace([np.inf, -np.inf], np.nan).dropna().values
    true_vals = results_df[true_col].astype(float).replace([np.inf, -np.inf], np.nan).dropna().values if true_col in results_df.columns else np.array([])

    row = {
        "sample": DATASET_PREFIX,
        "target": col,
        "n_blocks": int(len(vals)),
        "pred_mean": float(np.mean(vals)) if len(vals) else np.nan,
        "pred_median": float(np.median(vals)) if len(vals) else np.nan,
        "pred_std": float(np.std(vals, ddof=1)) if len(vals) > 1 else np.nan,
        "pred_min": float(np.min(vals)) if len(vals) else np.nan,
        "pred_max": float(np.max(vals)) if len(vals) else np.nan,
        "proxy_true_mean": float(np.mean(true_vals)) if len(true_vals) else np.nan,
        "proxy_true_median": float(np.median(true_vals)) if len(true_vals) else np.nan,
    }
    summary_rows.append(row)

whole_summary_df = pd.DataFrame(summary_rows)
whole_summary_path = PRED_OUT / f"{DATASET_PREFIX}_whole_sample_summary.csv"
whole_summary_xlsx = PRED_OUT / f"{DATASET_PREFIX}_whole_sample_summary.xlsx"
whole_summary_df.to_csv(whole_summary_path, index=False, encoding="utf-8-sig")
whole_summary_df.to_excel(whole_summary_xlsx, index=False)
print("整块代表性预测结果汇总已保存:", whole_summary_path)
display(whole_summary_df)


## 输出文件位置

运行完成后重点查看：

- `PM_GNN_prediction_results/EF-78_PM_GNN_prediction_results.csv`
- `PM_GNN_prediction_results/metrics_gnn_complete_academic_pm_gnn_EF-78.csv`
- `PM_GNN_prediction_results/EF-78_whole_sample_summary.csv`
- `PM_GNN_prediction_results/true_vs_predicted_*.png`
- `PM_GNN_prediction_results/figure_data/true_vs_predicted_*_data.csv`

以上文件就是 `EF-78.raw` 页岩数据的 PM-GNN 预测结果、指标表、整块代表性汇总、六张图以及对应绘图数据。
